# YEJOO 재고 수불부 STEP 분리 실행 노트북 (V16 기반)

- 셀 0: 유틸/함수/클래스 정의 (한 번만 실행)
- 셀 1: 공통 파일 선택/저장 함수
- STEP0: 원본 전처리만 별도
- STEP4: (기존 STEP1~3) 통합

각 STEP는 입력 파일 3개를 다시 선택해서 실행합니다.

In [4]:
import re
import time
import numpy as np
import pandas as pd
import math
import os
import warnings

warnings.filterwarnings("ignore")

from pathlib import Path
from datetime import datetime
from tkinter import Tk, filedialog
 
from bisect import bisect_left
from collections import defaultdict

from typing import Optional, List, Dict, Tuple


# -*- coding: utf-8 -*-
# =========================================================
# ✅ 옵션/컬럼 (사용자 제공 전역변수)
# =========================================================
WAREHOUSE_KEYWORD = "본사"
BONDED_WH_KEYWORD = "07보세창고"  # 보세 수불부에서 창고명

# --- STEP0 조정전표 식별 ---
ADJ_PARTY_VALUE = "[조정]"  # 거래처명 값이 이거면 조정전표

DATE_COL = "일자"
ITEM_COL = "품목코드"
NAME_COL = "품목명"
STOCK_COL = "재고수량"
WH_COL = "창고명"
PARTY_COL = "거래처명"
UNIT_COL = "단위"

OUT_QTY_COL = "출고수량"
OUT_PRICE_COL = "출고단가"

# 제출본(실-전) 컬럼
SUBMIT_NAME_COL = "품목명"
SUBMIT_USE_COL = "용도"
SUBMIT_STOCK_COL = "본사전산"
SUBMIT_REAL_COL = "본사실재고"
SUBMIT_DIFF_COL = "본사실-전"   # ✅ 표시용: 실재고 - 전산  (예: (4) = -4)

# STEP00 (입고 날짜 당김)
IN_QTY_COL_CANDIDATES = ["입고수량"]
STEP00_WORST_THRESHOLD = -1  # ✅ -20 이하만 대상(= -20, -21, -100 ...)

BONDED_KEYWORDS = ["보세"]     # 거래처명/전표문구에 "보세" 들어가면 보세이동으로 간주
MOVE_PREFIX = "[이동]"        # "[이동] 07보세창고 → 01본사창고" 같은 형태


# =========================================================
# ✅ 튜닝 파라미터
# =========================================================
MAX_ITEM_SECONDS = 500
MAX_ITEM_ATTEMPTS = 1500
MAX_CANDIDATES_PER_ITEM = 1800
PRINT_EVERY_N_ITEMS = 50

TOLS = [0.20, 0.30, 0.35]
ALLOW_SAME_MONTH_FALLBACK = True
DAYS_RULE_FIRST = 30

# STEP5 (단가 조정)
PROMO_PAID_MIN_QTY = 5
STEP5_MIN_QTY = 5
STEP5_MAX_PRICE_DIFF = 0.20

# =========================================================
# ✅ STEP00 월넘김/보세 룰 옵션
# =========================================================
STEP00_ALLOW_CROSS_MONTH = True
STEP00_MAX_PULL_DAYS = 45   # 당김 허용 일수 확대
STEP00_MAX_PULL_MONTHS = 4  # 당김 허용 월수 확대
STEP00_USE_BONDED_RULE = True

# ===== 전표 수정 최소화 핵심 파라미터 =====
IGNORE_WORST_ABS_QTY = 2
IGNORE_WORST_RATIO  = 0.005

# STEP0: 1/1 조정은 "완전해결"일 때만
STEP0_REQUIRE_FULL_RESOLVE = False

# STEP00 : 전표 1건으로 충분히 해결될 때만 사용
STEP00_RESOLVE_RATIO_MIN = 0.1  # 더 작은 해결도 STEP00에서 허용

# STEP0-1 : 대여 적용
STEP01_WORST_THRESHOLD = -10 #마이너스나는 수량
STEP01_MAX_FUTURE_MONTHS = 3 # 미룰수 있는 달수
STEP01_MIN_BIG_SALE_RATIO = 1.6 # 큰 출고 조건 2배일것
STEP01_EXCLUDE_PARTY_KEYWORDS = ("달랏마트",)

# ===== 중복 제거(일자+품목코드+거래처+입출고방향) =====  ***수불부 훼손되는 중복 제거임 Fasle로 미사용 필수***
DEDUP_SUFUL_DUPLICATES = False

# STEP00에서 [이동] 전표 사용 금지 (보세이동만 예외)
STEP00_BLOCK_NON_BONDED_MOVES = True

# STEP5 run 회복 제한
STEP5_RUN_MIN = -21
STEP5_RUN_MIN_RECOVER_RATIO = 0.3



def resort_within_day_and_recalc(df: pd.DataFrame, in_qty_col: str) -> pd.DataFrame:
    """
    품목별로:
    1) 날짜 오름차순 유지
    2) 같은 날짜 내: 입고 먼저 -> 출고 나중 (그 외는 중간)
    3) 마지막 tie-break: _uid(없으면 원래 순서)로 안정화
    4) 정렬된 결과로 재고 재계산
    """
    df = df.copy()

    # tie-break 키 보장
    if "_uid" in df.columns:
        df["_tie"] = pd.to_numeric(df["_uid"], errors="coerce").fillna(0).astype(int)
    else:
        df["_tie"] = np.arange(len(df), dtype=int)

    # 정렬 우선순위
    df["_pri"] = df.apply(lambda r: _inout_priority(r, in_qty_col), axis=1)

    parts = []
    for code, g in df.groupby(ITEM_COL, sort=False):
        g = g.copy()

        # ✅ 핵심: 날짜는 그대로, 같은 날짜만 재정렬
        g = g.sort_values([DATE_COL, "_pri", "_tie"], ascending=[True, True, True], kind="mergesort").reset_index(drop=True)

        # ✅ 정렬 후 재고 재계산
        g = recalc_stock_simple(g, in_qty_col)

        parts.append(g)

    out = pd.concat(parts, ignore_index=True)

    out = out.drop(columns=["_pri", "_tie"], errors="ignore")
    return out



# =========================================================
# PriceBook
# =========================================================
class PriceBook:
    def __init__(self, df_adj: pd.DataFrame):
        d = df_adj[[ITEM_COL, DATE_COL, OUT_QTY_COL, OUT_PRICE_COL, PARTY_COL]].copy()
        d[ITEM_COL] = d[ITEM_COL].astype(str)
        d[OUT_QTY_COL] = pd.to_numeric(d[OUT_QTY_COL], errors="coerce").fillna(0.0)
        d[OUT_PRICE_COL] = pd.to_numeric(d[OUT_PRICE_COL], errors="coerce").fillna(0.0)
        d[PARTY_COL] = d[PARTY_COL].astype(str).fillna("")

        d = d[(d[OUT_QTY_COL] > 0) & (d[OUT_PRICE_COL] > 0)].copy()
        d = d.sort_values([ITEM_COL, DATE_COL], kind="mergesort").reset_index(drop=True)

        book = {}
        if len(d) > 0:
            g = d.groupby([ITEM_COL, DATE_COL], as_index=False).last()
            for code, sub in g.groupby(ITEM_COL, sort=False):
                dates = pd.to_datetime(sub[DATE_COL], errors="coerce").to_numpy(dtype="datetime64[ns]")
                prices = sub[OUT_PRICE_COL].to_numpy(dtype=float)
                parties = sub[PARTY_COL].astype(str).to_numpy(dtype=object)
                book[str(code)] = (dates, prices, parties)
        self.book = book

    def _nearest_past_in_range(self, dates, prices, parties, start, end):
        idx_end = np.searchsorted(dates, end, side="right") - 1
        if idx_end < 0:
            return (np.nan, pd.NaT, "")
        if dates[idx_end] < start:
            return (np.nan, pd.NaT, "")
        return (float(prices[idx_end]), pd.Timestamp(dates[idx_end]), str(parties[idx_end]))

    def out_ref_D7_same_month(self, code: str, target_date: pd.Timestamp, days: int = 7):
        tup = self.book.get(str(code))
        if tup is None:
            return (np.nan, pd.NaT, "")
        dates, prices, parties = tup
        end = pd.Timestamp(target_date).to_datetime64()
        month_start = np.datetime64(pd.Timestamp(target_date.year, target_date.month, 1))
        d7_start = (pd.Timestamp(target_date) - pd.Timedelta(days=days)).to_datetime64()
        start = month_start if month_start > d7_start else d7_start
        return self._nearest_past_in_range(dates, prices, parties, start, end)

    def out_ref_M(self, code: str, target_date: pd.Timestamp):
        tup = self.book.get(str(code))
        if tup is None:
            return (np.nan, pd.NaT, "")
        dates, prices, parties = tup
        start = np.datetime64(pd.Timestamp(target_date.year, target_date.month, 1))
        end = pd.Timestamp(target_date).to_datetime64()
        return self._nearest_past_in_range(dates, prices, parties, start, end)



# =========================================================
# FastIndex (df.index == 0..n-1 이어야 함!)
# =========================================================
class FastIndex:
    def __init__(self, df_adj: pd.DataFrame):
        self.df = df_adj  # DF 저장
        self.item = self.df[ITEM_COL].astype(str).to_numpy(dtype=object)  # 품목코드 배열
        self.dates = pd.to_datetime(self.df[DATE_COL], errors="coerce").to_numpy(dtype="datetime64[ns]")  # 날짜 배열
        self.stock = pd.to_numeric(self.df[STOCK_COL], errors="coerce").fillna(0).to_numpy(dtype=float, copy=True)  # ✅ 쓰기 가능 복사본


        pos_map = {}
        for i, code in enumerate(self.item):
            pos_map.setdefault(str(code), []).append(i)
        self.pos_map = {k: np.array(v, dtype=int) for k, v in pos_map.items()}

        mono_map = {}
        date_map = {}
        for code, pos in self.pos_map.items():
            d = self.dates[pos]
            date_map[code] = d
            mono_map[code] = bool(np.all(d[1:] >= d[:-1])) if len(d) >= 2 else True
        self.date_map = date_map
        self.mono_map = mono_map

    def positions(self, code: str):
        return self.pos_map.get(str(code))

    def _first_pos_idx_by_date(self, code: str, date: pd.Timestamp):
        code = str(code)
        pos = self.positions(code)
        if pos is None or len(pos) == 0:
            return None

        d = self.date_map[code]
        target = pd.Timestamp(date).to_datetime64()

        if self.mono_map.get(code, True):
            j = int(np.searchsorted(d, target, side="left"))
            if j >= len(d):
                return None
            return j
        else:
            for j in range(len(d)):
                if d[j] >= target:
                    return j
            return None

    def min_stock_after(self, code: str, date: pd.Timestamp) -> float:
        code = str(code)
        pos = self.positions(code)
        if pos is None:
            return np.nan
        j = self._first_pos_idx_by_date(code, date)
        if j is None:
            return np.nan
        sel = pos[j:]
        return float(np.min(self.stock[sel])) if len(sel) else np.nan

    def apply_plus_after(self, code: str, date: pd.Timestamp, plus_qty: int):
        code = str(code)
        pos = self.positions(code)
        if pos is None:
            return
        j = self._first_pos_idx_by_date(code, date)
        if j is None:
            return
        sel = pos[j:]
        self.stock[sel] = self.stock[sel] + float(plus_qty)

    def apply_minus_after(self, code: str, date: pd.Timestamp, minus_qty: int):
        code = str(code)
        pos = self.positions(code)
        if pos is None:
            return
        j = self._first_pos_idx_by_date(code, date)
        if j is None:
            return
        sel = pos[j:]
        self.stock[sel] = self.stock[sel] - float(minus_qty)

    def apply_adjustment_inplace(self, target_code: str, target_date: pd.Timestamp, moves: list):
        plus_qty = int(sum(int(m["이동량"]) for m in moves))
        if plus_qty <= 0:
            return
        self.apply_plus_after(str(target_code), target_date, plus_qty)
        for m in moves:
            sub = str(m["대체품목코드"])
            qty = int(m["이동량"])
            self.apply_minus_after(sub, target_date, qty)


def reorder_and_recalc_inventory(df: pd.DataFrame, only_codes: set | None = None) -> pd.DataFrame:
    """
    (정렬 + 재고 재계산)
    - 기본: 품목코드, 일자, _row_id로 안정 정렬 후 품목별 러닝 재고를 다시 계산.
    - only_codes 제공 시: 이미 정렬된 df를 가정하고, 해당 품목들만 재고를 다시 계산(속도 개선).
      (품목별 재고는 다른 품목과 독립이므로 안전)
    """
    ensure_cols(df, [COL_CODE, COL_DATE, COL_STOCK, COL_IN_QTY, COL_OUT_QTY])

    out_df = df.copy()

    if "_row_id" not in out_df.columns:
        out_df["_row_id"] = np.arange(len(out_df), dtype=np.int64)

    if only_codes is None:
        out_df = out_df.sort_values([COL_CODE, COL_DATE, "_row_id"], kind="mergesort").reset_index(drop=True)
    else:
        # 정렬은 유지(재정렬 안함). 필요한 품목만 계산.
        if not isinstance(only_codes, set):
            only_codes = set(only_codes)

    code_arr = out_df[COL_CODE].astype(str).to_numpy()
    in_arr = to_num(out_df[COL_IN_QTY]).fillna(0.0).to_numpy(dtype=np.float64)
    out_arr = to_num(out_df[COL_OUT_QTY]).fillna(0.0).to_numpy(dtype=np.float64)
    stock_arr = to_num(out_df[COL_STOCK]).fillna(0.0).to_numpy(dtype=np.float64)

    n = len(out_df)
    if n == 0:
        return out_df

    # group boundaries
    # codes already sorted when only_codes is None; when only_codes given we assume still sorted.
    boundaries = np.flatnonzero(code_arr[1:] != code_arr[:-1]) + 1
    starts = np.r_[0, boundaries]
    ends = np.r_[boundaries, n]

    for s, e in zip(starts, ends):
        code = code_arr[s]
        if only_codes is not None and code not in only_codes:
            continue

        # anchor: 첫 행의 재고수량을 그대로 두고 다음부터 누적
        for i in range(s + 1, e):
            stock_arr[i] = stock_arr[i - 1] + in_arr[i] - out_arr[i]

    out_df[COL_STOCK] = stock_arr
    return out_df

def apply_step0_adjustments(df: pd.DataFrame, history: list, max_move_per_item_default: int = 20, verbose: bool = True):
    """
    STEP0: '조정' 전표(브라켓 조정/조정거래처)를 이용해 중간 마이너스(일시적 재고 음수)를 완화.
    - 핵심 규칙: 품목별로 조정 전표로 "이동(=출고 감액)"할 수 있는 총량(잔량)을 추적한다.
      예) 최대 20개 중 14개를 이동했다면 이후엔 6개만 더 이동 가능.
    - 기존 로직을 유지하되, 'cap(최대 이동량)'을 한 번에 초기화하지 않고 품목별 잔량을 차감하며 사용한다.
    """
    # 0) 전처리/검증
    ensure_cols(df, [COL_CODE, COL_DATE, COL_STOCK, COL_IN_QTY, COL_OUT_QTY, COL_CUST])
    df = df.copy()

    # 안정 정렬을 위해 row_id 보장
    if "_row_id" not in df.columns:
        df["_row_id"] = np.arange(len(df), dtype=np.int64)

    # 1) 브라켓 조정 플래그/조정풀 구성
    idx = build_fast_index(df)

    # 조정 전표 후보: 거래처가 ADJ_VENDOR 이거나, 브라켓 조정 플래그가 True인 행
    adj_mask = df[COL_CUST].astype(str).str.strip().eq(ADJ_VENDOR)
    if "_is_bracket_adj" in df.columns:
        adj_mask = adj_mask | df["_is_bracket_adj"].fillna(False)

    # 조정 전표 중 "출고"를 감액하는 방식이므로, 출고수량>0 인 것만 풀로
    out = to_num(df[COL_OUT_QTY]).fillna(0.0)
    adj_mask = adj_mask & (out > 0)

    adj_rows_by_item: dict[str, list[int]] = {}
    for i in np.flatnonzero(adj_mask.to_numpy()):
        code = df.at[i, COL_CODE]
        adj_rows_by_item.setdefault(code, []).append(int(i))

    # 품목별 이동 가능 잔량(cap_remaining) 초기화:
    # - 기본 max_move_per_item_default(=20)
    # - 단위가 EA면 cap을 더 크게(기존 로직: 50) 허용하되, 사용자 요구(최대 20)가 있으면
    #   max_move_per_item_default로 강제하고 싶으면 아래 EA_CAP을 동일 값으로 바꾸면 됨.
    EA_CAP = 50

    cap_remaining: dict[str, float] = {}
    for code, rows in adj_rows_by_item.items():
        unit = str(df.at[rows[0], COL_UNIT]) if COL_UNIT in df.columns else ""
        cap = EA_CAP if unit.strip().upper() == "EA" else max_move_per_item_default
        cap_remaining[code] = float(cap)

    # 2) 현재 재고 재계산(정렬/러닝스톡)
    df = reorder_and_recalc_inventory(df)

    # 3) 마이너스 런 탐지 후 조정 전표 출고 감액으로 보정
    touched_items: set[str] = set()
    neg_runs = find_mid_negative_runs(df)

    if verbose:
        print(f"[STEP0] mid-negative runs: {len(neg_runs)}")

    # 품목별로 런을 시간순으로 처리
    for run in neg_runs:
        code = run["code"]
        if code not in adj_rows_by_item:
            continue
        if cap_remaining.get(code, 0.0) <= 0:
            continue

        start_i = run["start_i"]   # 음수 시작 행 index
        end_i = run["end_i"]       # 음수 종료 행 index
        need_qty = float(run["need_qty"])  # 최소 필요량(음수 최저점의 절대값)

        if need_qty <= 0:
            continue

        # 남은 잔량만큼만 사용
        allowed = min(need_qty, cap_remaining[code])
        if allowed <= 0:
            continue

        # 런 시작 이전(또는 동일일자 이전)의 조정 전표에서 출고를 감액해 재고를 남긴다.
        run_date = df.at[start_i, COL_DATE]
        candidates = [i for i in adj_rows_by_item[code] if df.at[i, COL_DATE] <= run_date]
        if not candidates:
            continue

        # 최신 조정부터(런에 가까운 날짜부터) 사용하는 것이 일반적으로 영향 최소
        candidates.sort(key=lambda i: (df.at[i, COL_DATE], df.at[i, "_row_id"]), reverse=True)

        moved_total = 0.0
        for adj_i in candidates:
            if moved_total >= allowed:
                break
            cur_out = float(to_num(pd.Series([df.at[adj_i, COL_OUT_QTY]])).iloc[0] or 0.0)
            if cur_out <= 0:
                continue
            take = min(cur_out, allowed - moved_total)
            if take <= 0:
                continue

            # 출고 감액(= take 만큼 재고를 남김)
            df.at[adj_i, COL_OUT_QTY] = cur_out - take
            moved_total += take

        if moved_total > 0:
            cap_remaining[code] -= moved_total
            touched_items.add(code)
            history.append({
                "step": "STEP0",
                "code": code,
                "run_start_idx": int(start_i),
                "run_end_idx": int(end_i),
                "moved_qty": float(moved_total),
                "cap_remaining": float(cap_remaining[code]),
            })

    # 4) 보정된 품목만 재계산(전체 재계산은 느리므로)
    if touched_items:
        df = reorder_and_recalc_inventory(df, only_codes=touched_items)

    return df

def ensure_cols(df: pd.DataFrame, cols: list, ctx=""):
    miss = [c for c in cols if c not in df.columns]
    if miss:
        raise ValueError(f"{ctx} 필요한 컬럼 누락: {miss}")

def normalize_item_code(code: str) -> str:
    s = str(code).strip()
    s = re.sub(r"\.0$", "", s)
    if re.fullmatch(r"\d+", s):
        s = s.lstrip("0")
        return s if s else "0"
    return s

def safe_str(x) -> str:
    if x is None:
        return ""
    if isinstance(x, float) and np.isnan(x):
        return ""
    s = str(x)
    return "" if s.lower() == "nan" else s

def parse_num(x):
    """콤마/괄호음수(예: (4))까지 안전 파싱"""
    if x is None:
        return 0.0
    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return 0.0
    s = s.replace(",", "")
    if re.fullmatch(r"\(\s*[-+]?\d+(\.\d+)?\s*\)", s):
        s = "-" + re.sub(r"[()\s]", "", s)
    v = pd.to_numeric(s, errors="coerce")
    return float(v) if not pd.isna(v) else 0.0

def to_float0(x) -> float:
    try:
        v = pd.to_numeric(x, errors="coerce")
        if pd.isna(v):
            return 0.0
        return float(v)
    except Exception:
        return 0.0

def is_taxfree(name: str) -> bool:
    s = str(name) if name is not None else ""
    return "(특)" in s

def pick_party(df_item: pd.DataFrame, date: pd.Timestamp) -> str:
    d = df_item[df_item[DATE_COL] == date]
    if len(d) == 0:
        return ""
    v = str(d.iloc[-1].get(PARTY_COL, "")).strip()
    return v if v and v.lower() != "nan" else ""

def month_range(year: int, month: int):
    start = pd.Timestamp(year, month, 1)
    end = (start + pd.offsets.MonthEnd(1)).normalize()
    return start, end

def detect_in_qty_col(df: pd.DataFrame) -> str:
    for c in IN_QTY_COL_CANDIDATES:
        if c in df.columns:
            return c
    for c in df.columns:
        if "입고" in str(c):
            return c
    raise ValueError(f"[입고컬럼] 입고수량 컬럼을 못 찾음. 후보={IN_QTY_COL_CANDIDATES} / 실제 컬럼={list(df.columns)}")

def is_move_voucher(party: str) -> bool:
    s = safe_str(party)
    return "[이동]" in s

def is_bonded_to_hq_move(party: str) -> bool:
    """[이동] 07보세창고 → 01본사창고 만 True"""
    s = safe_str(party)
    if "[이동]" not in s:
        return False
    return ("07보세창고" in s) and ("01본사창고" in s)

def is_bonded_move(party: str) -> bool:
    s = safe_str(party)
    if not s:
        return False
    if any(k in s for k in BONDED_KEYWORDS):
        return True
    if s.startswith(MOVE_PREFIX) and "보세" in s:
        return True
    return False

def read_excel_best_sheet(path: str, include_keywords=None):
    """
    - include_keywords: 예) ["수불부"] 또는 ["제출본", "실-전"]
    - 키워드 포함 시트가 있으면 그 중 첫 번째를 읽고, 없으면 첫 시트.
    """
    xls = pd.ExcelFile(path)
    sheets = xls.sheet_names
    if include_keywords:
        for sh in sheets:
            nm = str(sh)
            if all(k in nm for k in include_keywords):
                return pd.read_excel(path, sheet_name=sh), sh
        for sh in sheets:
            nm = str(sh)
            if any(k in nm for k in include_keywords):
                return pd.read_excel(path, sheet_name=sh), sh
    return pd.read_excel(path, sheet_name=sheets[0]), sheets[0]

def reorder_within_same_day_by_stock_chain(df: pd.DataFrame, in_qty_col: str) -> pd.DataFrame:
    """
    ✅ 같은 (품목코드, 일자) 안에서 순서가 깨졌을 때 복원 + 재고 재계산.

    왜 필요?
    - 중복제거/전표삽입/날짜이동/정렬 등 과정에서 "같은 날짜" 내 행 순서가 바뀌면,
      STOCK_COL(재고수량) 흐름이 깨져 중간 마이너스(run) 탐지/해결이 왜곡될 수 있음.

    복원 방식(최소 변경 지향):
    1) (품목코드)별 DATE_COL 오름차순
    2) 같은 날짜는 "prev_close(전일 마감 재고)"에서 (입고-출고) 델타를 적용했을 때
       해당 행의 STOCK_COL과 일치하는 행을 greedy로 이어붙임.
       - 매칭 실패 시: 기존 _orig_order(원본 순서) 우선으로 유지.
    3) 최종적으로 recalc_stock_item_with_anchor(앵커/조정 규칙 포함)로 재고 재계산.

    ⚠️ 주의:
    - 이 함수는 '순서/재고'만 정합화하는 용도이며, 전표의 수량/날짜 변경 자체는 각 STEP 로직에서 수행.
    """

    df = df.copy()

    # 안정 정렬 기준(처음 생성된 원본 순서를 최대한 유지)
    if "_orig_order" not in df.columns:
        df["_orig_order"] = np.arange(len(df), dtype=int)
    else:
        df["_orig_order"] = pd.to_numeric(df["_orig_order"], errors="coerce").fillna(0).astype(int)

    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce", format="mixed")
    df[in_qty_col] = pd.to_numeric(df[in_qty_col], errors="coerce").fillna(0).astype(float)
    df[OUT_QTY_COL] = pd.to_numeric(df[OUT_QTY_COL], errors="coerce").fillna(0).astype(float)
    df[STOCK_COL] = pd.to_numeric(df[STOCK_COL], errors="coerce").fillna(0).astype(float)
    df[PARTY_COL] = df[PARTY_COL].astype(str).fillna("")

    def _reorder_one_day(g_day: pd.DataFrame, prev_close: Optional[float]):
        g = g_day.copy()
        g = g.sort_values(["_orig_order"], kind="mergesort").reset_index(drop=True)
        g["_in"] = g[in_qty_col].astype(float)
        g["_out"] = g[OUT_QTY_COL].astype(float)
        g["_stk"] = g[STOCK_COL].astype(float)

        tol = 1e-6
        if prev_close is None:
            # 전일 마감이 없으면 첫 행 stock을 시작점으로 둠(가장 보수적)
            current = float(g["_stk"].iloc[0]) if len(g) else 0.0
        else:
            current = float(prev_close)

        remaining = g.copy()
        ordered = []

        while len(remaining) > 0:
            expected = current + (remaining["_in"] - remaining["_out"]).to_numpy(dtype=float)
            diff = np.abs(expected - remaining["_stk"].to_numpy(dtype=float))
            hit = np.where(diff <= tol)[0]
            if len(hit) > 0:
                pick = int(hit[0])
                row = remaining.iloc[pick:pick+1].copy()
                ordered.append(row)
                current = float(row["_stk"].iloc[0])  # stock 힌트를 신뢰
                remaining = pd.concat([remaining.iloc[:pick], remaining.iloc[pick+1:]], axis=0).reset_index(drop=True)
                continue

            # 매칭 실패: 원본 순서대로 1행 채택(최소 변경)
            row = remaining.iloc[:1].copy()
            ordered.append(row)
            # stock 힌트는 불신 -> expected로만 current 진행(뒤에서 재계산으로 정합화)
            current = current + float(row["_in"].iloc[0]) - float(row["_out"].iloc[0])
            remaining = remaining.iloc[1:].reset_index(drop=True)

        out = pd.concat(ordered, ignore_index=True)
        out = out.drop(columns=["_in", "_out", "_stk"], errors="ignore")
        return out

    parts = []
    for code, g_item in df.groupby(ITEM_COL, sort=False):
        g_item = g_item.sort_values([DATE_COL, "_orig_order"], kind="mergesort").copy()

        out_days = []
        prev_close = None

        for d, g_day in g_item.groupby(DATE_COL, sort=False):
            g_day2 = _reorder_one_day(g_day, prev_close)
            out_days.append(g_day2)

            # 전일 마감(힌트): 당일 마지막 행 stock
            try:
                prev_close = float(pd.to_numeric(g_day2[STOCK_COL], errors="coerce").fillna(0).iloc[-1])
            except Exception:
                pass

        g2 = pd.concat(out_days, ignore_index=True)
        g2 = g2.sort_values([DATE_COL, "_orig_order"], kind="mergesort").reset_index(drop=True)

        # ✅ 최종 재고 재계산(앵커 포함)
        g2 = recalc_stock_item_with_anchor(g2, in_qty_col)

        parts.append(g2)

    out = pd.concat(parts, ignore_index=True)
    out = out.sort_values([ITEM_COL, DATE_COL, "_orig_order"], kind="mergesort").reset_index(drop=True)
    return out

def remove_yearly_duplicate_blocks_strict_v2(df: pd.DataFrame, min_block_len: int = 200):
    df = df.copy()
    need_cols = [ITEM_COL, DATE_COL, STOCK_COL, OUT_QTY_COL, OUT_PRICE_COL, PARTY_COL]
    ensure_cols(df, need_cols, ctx="[중복제거]")

    logs = []
    drop_idx = []

    for code, g in df.groupby(ITEM_COL, sort=False):
        gg = g.copy()
        dates = gg[DATE_COL].to_numpy(dtype="datetime64[ns]")
        if len(dates) < (min_block_len * 2):
            continue

        reset_pos = np.where(dates[1:] < dates[:-1])[0]
        if len(reset_pos) == 0:
            continue

        cut = int(reset_pos[0] + 1)
        if cut < min_block_len:
            continue

        left = gg.iloc[:cut].copy()
        right = gg.iloc[cut:].copy()
        if len(right) < min_block_len:
            continue

        def _sig(d: pd.DataFrame):
            x = d[[DATE_COL, STOCK_COL, OUT_QTY_COL, OUT_PRICE_COL, PARTY_COL]].copy()
            x[PARTY_COL] = x[PARTY_COL].astype(str).fillna("")
            x = x.fillna(0)
            return list(
                zip(
                    x[DATE_COL].astype("datetime64[ns]").to_numpy(),
                    x[STOCK_COL].astype(float).to_numpy(),
                    x[OUT_QTY_COL].astype(float).to_numpy(),
                    x[OUT_PRICE_COL].astype(float).to_numpy(),
                    x[PARTY_COL].astype(str).to_numpy(),
                )
            )

        sigL = _sig(left)
        sigR = _sig(right)
        n = len(sigL)

        if len(sigR) >= n and sigR[:n] == sigL:
            dup_part = right.iloc[:n]
            drop_idx.extend(dup_part.index.tolist())
            logs.append(
                {
                    ITEM_COL: str(code),
                    "원본행수": int(len(gg)),
                    "최종행수": int(len(gg) - len(dup_part)),
                    "제거행수": int(len(dup_part)),
                    "사유": f"date reset 기반 1년치 중복 제거 (cut={cut}, drop={len(dup_part)})",
                }
            )

    df_clean = df.drop(index=drop_idx).sort_index()
    rep = pd.DataFrame(logs)
    if len(rep) == 0:
        rep = pd.DataFrame(columns=[ITEM_COL, "원본행수", "최종행수", "제거행수", "사유"])
    else:
        rep = rep.sort_values("제거행수", ascending=False).reset_index(drop=True)

    return df_clean, rep

def remove_non_product_items(df: pd.DataFrame):
    df = df.copy()
    df[NAME_COL] = df[NAME_COL].astype(str)

    EXCLUDE_KEYWORDS = [
        "택배", "배송",
        "과세", "면세",
        "파렛트", "파레트", "팔레트", "파렛",
        "초록", "연두", "파랑", "빨강",
        "잡화비",
        "임대료", "렌탈", "대여",
        "관리비", "수수료",
        "CHEF", "HALES", "비타민",
    ]
    pattern = "|".join(EXCLUDE_KEYWORDS)
    mask = df[NAME_COL].str.contains(pattern, regex=True, na=False)

    removed = df[mask].copy()
    df_clean = df[~mask].copy()

    rep = (
        removed.groupby(ITEM_COL, sort=False)
        .agg(품목명=(NAME_COL, "first"), 제거행수=(ITEM_COL, "size"))
        .reset_index()
        .sort_values("제거행수", ascending=False)
    )
    return df_clean, rep

def dedup_by_date_item_party_direction(df: pd.DataFrame, in_qty_col: str):
    df = df.copy()
    ensure_cols(df, [DATE_COL, ITEM_COL, PARTY_COL, OUT_QTY_COL, in_qty_col], ctx="[중복제거]")

    df["_dedup_in"] = pd.to_numeric(df[in_qty_col], errors="coerce").fillna(0).astype(float)
    df["_dedup_out"] = pd.to_numeric(df[OUT_QTY_COL], errors="coerce").fillna(0).astype(float)
    df["_dedup_dir"] = np.select(
        [
            (df["_dedup_in"] > 0) & (df["_dedup_out"] <= 0),
            (df["_dedup_out"] > 0) & (df["_dedup_in"] <= 0),
        ],
        ["IN", "OUT"],
        default="MIX",
    )

    # 안정적인 보존 순서
    df["_dedup_order"] = np.arange(len(df), dtype=int)

    before = int(len(df))
    df = df.sort_values([DATE_COL, ITEM_COL, PARTY_COL, "_dedup_dir", "_dedup_order"], kind="mergesort")
    df_clean = df.drop_duplicates(
        subset=[DATE_COL, ITEM_COL, PARTY_COL, "_dedup_dir"],
        keep="first",
    ).copy()

    removed = int(before - len(df_clean))
    rep = pd.DataFrame([
        {
            "원본행수": before,
            "최종행수": int(len(df_clean)),
            "제거행수": removed,
            "사유": "일자+품목코드+거래처+입출고방향 중복 제거",
        }
    ])

    df_clean = df_clean.drop(columns=["_dedup_in", "_dedup_out", "_dedup_dir", "_dedup_order"], errors="ignore")
    return df_clean, rep

def build_yearly_min_stock_table(df: pd.DataFrame):
    df_min = (
        df.groupby(ITEM_COL, sort=False)[STOCK_COL]
        .min()
        .reset_index()
        .rename(columns={STOCK_COL: "연중최소재고"})
    )
    min_map = {str(r[ITEM_COL]): float(r["연중최소재고"]) for _, r in df_min.iterrows()}
    return min_map, df_min

def build_runs_by_row_with_day_close_check(item_df: pd.DataFrame) -> list:
    stocks = pd.to_numeric(item_df[STOCK_COL], errors="coerce").fillna(0).to_numpy(dtype=float)
    dates = pd.to_datetime(item_df[DATE_COL], errors="coerce").to_numpy(dtype="datetime64[ns]")
    n = len(stocks)
    if n == 0:
        return []

    first_idx_by_date = {}
    close_stock_by_date = {}
    for i in range(n):
        d = dates[i]
        if d not in first_idx_by_date:
            first_idx_by_date[d] = i
        close_stock_by_date[d] = float(stocks[i])

    runs = []
    start_i = None

    def finalize_run(si, ei):  # run 확정 함수
        if ei < si:  # 역전이면
            return  # 종료
        seg = stocks[si : ei + 1]  # 구간 재고
        if len(seg) == 0:  # 비면
            return  # 종료
        first_neg_rel = None  # 최초 음수 위치(상대)
        for j in range(len(seg)):  # 구간 순회
            if float(seg[j]) < 0:  # 음수면
                first_neg_rel = j  # 최초 음수 저장
                break  # 종료
        if first_neg_rel is None:  # 음수 없으면
            return  # 종료
        first_neg = float(seg[first_neg_rel])  # 최초 음수 값
        need = int(abs(first_neg))  # ✅ 필요수량 = 최초 음수 절대값
        worst = float(np.min(seg))  # 최악값(참고용)
        first_neg_date = pd.Timestamp(dates[si + first_neg_rel])  # ✅ 최초 음수 발생일
        runs.append({
            "run_start_row": int(si),
            "run_end_row": int(ei),
            "run_start": first_neg_date,  # ✅ 최초 음수일을 run_start로
            "run_end": pd.Timestamp(dates[ei]),
            "worst_min": worst,
            "first_neg": first_neg,
            "first_neg_date": first_neg_date,
            "need_qty": need,
            "run_len": int(ei - si + 1),
        })

    for i in range(n):
        if stocks[i] < 0:
            if start_i is None:
                start_i = i
        else:
            if start_i is not None:
                end_i = i - 1
                while end_i >= start_i:
                    end_date = dates[end_i]
                    if close_stock_by_date.get(end_date, 0.0) >= 0:
                        new_end = first_idx_by_date[end_date] - 1
                        if new_end >= end_i:
                            end_i -= 1
                        else:
                            end_i = new_end
                        continue
                    break
                finalize_run(start_i, end_i)
                start_i = None

    if start_i is not None:
        end_i = n - 1
        while end_i >= start_i:
            end_date = dates[end_i]
            if close_stock_by_date.get(end_date, 0.0) >= 0:
                new_end = first_idx_by_date[end_date] - 1
                if new_end >= end_i:
                    end_i -= 1
                else:
                    end_i = new_end
                continue
            break
        finalize_run(start_i, end_i)

    return runs

def init_submit_state(df_submit: pd.DataFrame):
    sub = df_submit.copy()
    ensure_cols(sub, [ITEM_COL], ctx="[제출본]")
    sub[ITEM_COL] = sub[ITEM_COL].astype(str).apply(normalize_item_code)

    for c in [SUBMIT_NAME_COL, SUBMIT_USE_COL]:
        if c not in sub.columns:
            sub[c] = ""

    if SUBMIT_REAL_COL not in sub.columns:
        sub[SUBMIT_REAL_COL] = 0
    if SUBMIT_STOCK_COL not in sub.columns:
        sub[SUBMIT_STOCK_COL] = 0
    if SUBMIT_DIFF_COL not in sub.columns:
        sub[SUBMIT_DIFF_COL] = 0

    sub[SUBMIT_REAL_COL] = sub[SUBMIT_REAL_COL].apply(parse_num).astype(float)
    sub[SUBMIT_STOCK_COL] = sub[SUBMIT_STOCK_COL].apply(parse_num).astype(float)
    sub[SUBMIT_DIFF_COL] = (sub[SUBMIT_REAL_COL] - sub[SUBMIT_STOCK_COL]).astype(float)

    real_map = dict(zip(sub[ITEM_COL], sub[SUBMIT_REAL_COL]))
    sys_map = dict(zip(sub[ITEM_COL], sub[SUBMIT_STOCK_COL]))

    avail_map = {
        k: int(max(0, np.floor(float(sys_map.get(k, 0.0)) - float(real_map.get(k, 0.0)))))
        for k in real_map.keys()
    }

    sub[SUBMIT_USE_COL] = sub[SUBMIT_USE_COL].astype(str).str.strip()
    sub[SUBMIT_NAME_COL] = sub[SUBMIT_NAME_COL].astype(str)
    submit_map = sub.set_index(ITEM_COL, drop=False).to_dict("index")

    return sub, submit_map, real_map, sys_map, avail_map

def apply_sys_delta(sys_map: dict, real_map: dict, avail_map: dict, code: str, delta: int):
    code = str(code)
    sys_map[code] = float(sys_map.get(code, 0.0)) + float(delta)
    real = float(real_map.get(code, 0.0))
    sysv = float(sys_map.get(code, 0.0))
    avail_map[code] = int(max(0, np.floor(sysv - real)))

def materialize_submit_df(df_submit_template: pd.DataFrame, real_map, sys_map) -> pd.DataFrame:
    sub = df_submit_template.copy()
    sub[ITEM_COL] = sub[ITEM_COL].astype(str).apply(normalize_item_code)

    if SUBMIT_REAL_COL not in sub.columns:
        sub[SUBMIT_REAL_COL] = 0
    if SUBMIT_STOCK_COL not in sub.columns:
        sub[SUBMIT_STOCK_COL] = 0
    if SUBMIT_DIFF_COL not in sub.columns:
        sub[SUBMIT_DIFF_COL] = 0

    sub[SUBMIT_REAL_COL] = sub[ITEM_COL].map(real_map).fillna(0).astype(float)
    sub[SUBMIT_STOCK_COL] = sub[ITEM_COL].map(sys_map).fillna(0).astype(float)
    sub[SUBMIT_DIFF_COL] = (sub[SUBMIT_REAL_COL] - sub[SUBMIT_STOCK_COL]).astype(float)
    return sub

def subs_ok_after_fast(idx: FastIndex, moves: list, date: pd.Timestamp) -> bool:
    for m in moves:
        sub = str(m["대체품목코드"])
        qty = float(m["이동량"])
        smin = idx.min_stock_after(sub, date)
        if pd.isna(smin) or (float(smin) - qty) < 0:
            return False
    return True

def is_solved_after_fast(idx: FastIndex, code: str, date: pd.Timestamp, plus_qty: int) -> bool:
    smin = idx.min_stock_after(code, date)
    if pd.isna(smin):
        return True
    return (float(smin) + float(plus_qty)) >= 0

def build_moves_fast(  # 후보 move 생성 함수
    idx: FastIndex,  # FastIndex
    pricebook: PriceBook,  # PriceBook
    df_submit: pd.DataFrame,  # 제출본 DF
    submit_avail: dict,  # 제출본 가용맵
    yearly_min_stock_map: dict,  # 연중최소재고(미사용 유지)
    target_code: str,  # 대상 품목코드
    target_name: str,  # 대상 품목명
    target_use: str,  # 대상 실-전
    target_date: pd.Timestamp,  # 기준일
    need_qty: int,  # 필요수량
    tol: float,  # 단가 허용오차
    date_mode: str,  # 날짜모드(D7/M)
    allow_partial: bool,  # 부분허용 여부
):
    if date_mode == "D7":  # D7면
        base_price, _, _ = pricebook.out_ref_D7_same_month(target_code, target_date, days=DAYS_RULE_FIRST)  # 기준단가 찾기
    else:  # M이면
        base_price, _, _ = pricebook.out_ref_M(target_code, target_date)  # 기준단가 찾기

    if pd.isna(base_price) or base_price <= 0:  # 기준단가 없으면
        return [], "NO_BASE_PRICE"  # 종료

    target_taxfree = is_taxfree(target_name)  # 면세여부

    cand = df_submit.copy()  # 후보 복사
    cand[ITEM_COL] = cand[ITEM_COL].astype(str).apply(normalize_item_code)  # 품목코드 정리
    cand = cand[cand[ITEM_COL] != str(target_code)].copy()  # 자기자신 제외

    if str(target_use).strip() != "":  # 실-전 조건 있으면
        cand = cand[cand[SUBMIT_USE_COL].astype(str).str.strip() == str(target_use)].copy()  # 실-전 동일만

    cand["taxfree"] = cand[SUBMIT_NAME_COL].apply(is_taxfree)  # 후보 면세여부 계산
    cand = cand[cand["taxfree"] == target_taxfree].copy()  # 면세/과세 동일만

    codes = cand[ITEM_COL].astype(str).tolist()  # 코드 리스트
    cand["pool"] = [int(submit_avail.get(c, 0)) for c in codes]  # 제출본 가용 풀
    cand = cand[cand["pool"] > 0].copy()  # 풀 없는건 제거
    if len(cand) == 0:  # 후보 없으면
        return [], "NO_CAND"  # 종료

    if len(cand) > MAX_CANDIDATES_PER_ITEM:  # 후보 많으면
        cand = cand.sort_values(["pool"], ascending=[False]).head(MAX_CANDIDATES_PER_ITEM).copy()  # 풀 큰순 자르기

    ref_prices, ref_dates, ref_parties = [], [], []  # 참조값 배열
    for c in cand[ITEM_COL].astype(str).tolist():  # 후보 반복
        if date_mode == "D7":  # D7면
            p, d0, party0 = pricebook.out_ref_D7_same_month(c, target_date, days=DAYS_RULE_FIRST)  # D7 참조
        else:  # M이면
            p, d0, party0 = pricebook.out_ref_M(c, target_date)  # 월내 참조
        ref_prices.append(p)  # 단가 저장
        ref_dates.append(d0)  # 일자 저장
        ref_parties.append(party0)  # 거래처 저장

    cand["T_출고단가"] = np.array(ref_prices, dtype=float)  # 후보 출고단가
    cand["T_전표일자"] = ref_dates  # 후보 전표일자
    cand["T_거래처명"] = ref_parties  # 후보 거래처명

    cand = cand[~pd.isna(cand["T_출고단가"])].copy()  # 단가 없는거 제거
    cand = cand[cand["T_출고단가"] > 0].copy()  # 0단가 제거
    cand = cand[cand["T_거래처명"].astype(str).str.len() > 0].copy()  # 거래처 없는거 제거
    if len(cand) == 0:  # 후보 없으면
        return [], "NO_PRICE_REF"  # 종료

    cand["단가차이율"] = (cand["T_출고단가"] - float(base_price)).abs() / float(base_price)  # 차이율 계산
    cand = cand[cand["단가차이율"] <= float(tol)].copy()  # tol 이내만
    if len(cand) == 0:  # 후보 없으면
        return [], "NO_TOL"  # 종료

    mins = [idx.min_stock_after(str(c), target_date) for c in cand[ITEM_COL].astype(str).tolist()]  # 이후최소재고 계산
    cand["T이후_최소재고"] = np.array(mins, dtype=float)  # 컬럼 저장
    cand = cand[~pd.isna(cand["T이후_최소재고"])].copy()  # NaN 제거
    cand["재고_가용"] = cand["T이후_최소재고"].apply(lambda x: int(np.floor(max(0.0, float(x)))))  # 재고가용 계산
    cand = cand[cand["재고_가용"] > 0].copy()  # 재고가용 없는거 제거
    if len(cand) == 0:  # 후보 없으면
        return [], "NO_STOCK_AVAIL"  # 종료

    cand["최종가용"] = cand.apply(lambda r: int(min(int(r["재고_가용"]), int(r["pool"]))), axis=1)  # 최종가용 계산
    cand = cand[cand["최종가용"] > 0].copy()  # 최종가용 없는거 제거
    if len(cand) == 0:  # 후보 없으면
        return [], "NO_FINAL_AVAIL"  # 종료

    # =========================  # 구분선
    # ✅ move 수 감소 핵심 1) 정렬을 "최종가용 큰 것" 우선  # 설명
    # =========================  # 구분선
    cand = cand.sort_values(["최종가용", "단가차이율"], ascending=[False, True]).reset_index(drop=True)  # 큰거 먼저

    # =========================  # 구분선
    # ✅ move 수 감소 핵심 2) 단일 후보로 FULL 해결 먼저 시도  # 설명
    # =========================  # 구분선
    need = int(need_qty)  # 필요수량
    if not allow_partial:  # FULL 모드면
        one = cand[cand["최종가용"] >= need].copy()  # 한방 가능한 후보만
        if len(one) > 0:  # 있으면
            one = one.sort_values(["단가차이율"], ascending=[True]).reset_index(drop=True)  # 단가차이 최소
            sub_code = str(one.loc[0, ITEM_COL])  # 후보코드
            sub_name = str(one.loc[0, SUBMIT_NAME_COL])  # 후보명
            mv = int(need)  # 이동량(need만큼)
            moves = [  # moves 1개로 종료
                {
                    "대체품목코드": sub_code,  # 대체코드
                    "대체품목명": sub_name,  # 대체명
                    "이동량": mv,  # 이동량
                    "대체품목_날짜": pd.Timestamp(target_date),  # ✅ 대체 적용일자를 최초 음수일로 고정
                    "대체품목_거래처명": str(one.loc[0, "T_거래처명"]),  # 거래처
                }
            ]
            return moves, "OK"  # 단일 move 성공 반환

    # =========================  # 구분선
    # 기본 로직: 큰 후보부터 채우기(그래도 move 수 줄어듦)  # 설명
    # =========================  # 구분선
    moves = []  # move 리스트
    for i in range(len(cand)):  # 후보 반복
        if need <= 0:  # 다 채웠으면
            break  # 종료
        sub_code = str(cand.loc[i, ITEM_COL])  # 후보코드
        sub_name = str(cand.loc[i, SUBMIT_NAME_COL])  # 후보명
        avail = int(cand.loc[i, "최종가용"])  # 후보가용
        mv = int(min(need, avail))  # 이동량 계산
        if mv <= 0:  # 0이면
            continue  # 스킵
        moves.append(  # move 추가
            {
                "대체품목코드": sub_code,  # 대체코드
                "대체품목명": sub_name,  # 대체명
                "이동량": mv,  # 이동량
                "대체품목_날짜": pd.Timestamp(target_date),  # ✅ 최초 음수일로 고정
                "대체품목_거래처명": str(cand.loc[i, "T_거래처명"]),  # 거래처
            }
        )
        need -= mv  # 남은 need 감소

    if (need > 0) and (not allow_partial):  # FULL인데 모자라면
        return [], "NOT_ENOUGH"  # 실패
    return moves, ("PARTIAL" if need > 0 else "OK")  # 결과 반환

def one_shot_full_possible_fast(  # 단일 후보 FULL 가능 빠른 체크
    idx: FastIndex,  # FastIndex
    pricebook: PriceBook,  # PriceBook
    df_submit: pd.DataFrame,  # 제출본 DF
    submit_avail: dict,  # 제출본 가용맵
    target_code: str,  # 대상코드
    target_name: str,  # 대상명
    target_use: str,  # 실-전
    target_date: pd.Timestamp,  # 기준일
    need_qty: int,  # 필요수량
    tol: float,  # 단가 허용오차
    date_mode: str,  # 날짜모드(D7/M)
) -> bool:  # 가능 여부
    if int(need_qty) <= 0:  # need 없으면
        return False  # 불가

    if date_mode == "D7":  # D7면
        base_price, _, _ = pricebook.out_ref_D7_same_month(target_code, target_date, days=DAYS_RULE_FIRST)  # 기준단가
    else:  # M이면
        base_price, _, _ = pricebook.out_ref_M(target_code, target_date)  # 기준단가

    if pd.isna(base_price) or float(base_price) <= 0:  # 기준단가 없으면
        return False  # 불가

    target_taxfree = is_taxfree(target_name)  # 면세여부
    need = int(need_qty)  # need 정수

    cand = df_submit.copy()  # 후보 복사
    cand[ITEM_COL] = cand[ITEM_COL].astype(str).apply(normalize_item_code)  # 품목코드 정리
    cand = cand[cand[ITEM_COL] != str(target_code)].copy()  # 자기자신 제외

    if str(target_use).strip() != "":  # 실-전 조건 있으면
        cand = cand[cand[SUBMIT_USE_COL].astype(str).str.strip() == str(target_use)].copy()  # 동일만

    cand["taxfree"] = cand[SUBMIT_NAME_COL].apply(is_taxfree)  # 면세 계산
    cand = cand[cand["taxfree"] == target_taxfree].copy()  # 동일만
    if len(cand) == 0:  # 후보 없으면
        return False  # 불가

    codes = cand[ITEM_COL].astype(str).tolist()  # 코드 리스트
    cand["pool"] = [int(submit_avail.get(c, 0)) for c in codes]  # 제출본 가용 풀
    cand = cand[cand["pool"] > 0].copy()  # 풀 없는건 제거
    if len(cand) == 0:  # 후보 없으면
        return False  # 불가

    cand = cand.sort_values(["pool"], ascending=[False]).copy()  # pool 큰 순
    if len(cand) > MAX_CANDIDATES_PER_ITEM:  # 후보 많으면
        cand = cand.head(MAX_CANDIDATES_PER_ITEM).copy()  # 상위만

    # ✅ 후보를 하나씩 보면서 "한방 가능" 나오면 즉시 True(빠름)
    for _, r in cand.iterrows():  # 후보 반복
        sub_code = str(r[ITEM_COL])  # 후보코드
        sub_name = str(r.get(SUBMIT_NAME_COL, ""))  # 후보명(참고)

        if date_mode == "D7":  # D7면
            p, _, party0 = pricebook.out_ref_D7_same_month(sub_code, target_date, days=DAYS_RULE_FIRST)  # 단가참조
        else:  # M이면
            p, _, party0 = pricebook.out_ref_M(sub_code, target_date)  # 단가참조

        if pd.isna(p) or float(p) <= 0:  # 단가 없으면
            continue  # 다음
        if safe_str(party0) == "":  # 거래처 없으면
            continue  # 다음

        diff = abs(float(p) - float(base_price)) / float(base_price)  # 단가차이율
        if diff > float(tol):  # tol 초과면
            continue  # 다음

        smin = idx.min_stock_after(sub_code, target_date)  # 이후최소재고
        if pd.isna(smin):  # NaN이면
            continue  # 다음

        stock_avail = int(np.floor(max(0.0, float(smin))))  # 재고가용
        if stock_avail <= 0:  # 0이면
            continue  # 다음

        pool_avail = int(r.get("pool", 0))  # 풀가용
        final_avail = int(min(stock_avail, pool_avail))  # 최종가용
        if final_avail < need:  # need 못채우면
            continue  # 다음

        # ✅ 대체품목이 need만큼 빼도 0 미만 안되는지 최종 확인
        if float(smin) - float(need) < 0:  # 빼면 마이너스면
            continue  # 다음

        return True  # ✅ 한방 가능

    return False  # 한방 불가

def try_fix_once_fast(
    idx: FastIndex,
    pricebook: PriceBook,
    df_submit: pd.DataFrame,
    submit_avail: dict,
    yearly_min_stock_map: dict,
    target_code: str,
    target_name: str,
    target_use: str,
    target_date: pd.Timestamp,
    need_qty: int,
    run_tag: str,
    allow_partial: bool,
    force_date_modes: list = None,
):
    if force_date_modes is not None:
        date_modes = force_date_modes
    else:
        date_modes = ["D7"]
        if ALLOW_SAME_MONTH_FALLBACK:
            date_modes.append("M")

    # =========================  # 구분선
    # ✅ "단일 후보 FULL 가능" 빠른 체크 결과를 tag에 남기기  # 설명
    # =========================  # 구분선
    one_shot_hint = False  # 힌트 기본값
    if not allow_partial:  # FULL일 때만
        for dm in date_modes:  # 모드 반복
            for tol0 in sorted(TOLS, reverse=True):  # tol 큰값부터
                if one_shot_full_possible_fast(  # 빠른 체크
                    idx=idx,  # idx
                    pricebook=pricebook,  # pricebook
                    df_submit=df_submit,  # 제출본
                    submit_avail=submit_avail,  # 가용
                    target_code=target_code,  # 대상
                    target_name=target_name,  # 대상명
                    target_use=target_use,  # 실-전
                    target_date=target_date,  # 날짜
                    need_qty=need_qty,  # need
                    tol=tol0,  # tol
                    date_mode=dm,  # 모드
                ):
                    one_shot_hint = True  # 힌트 True
                    break  # tol 루프 종료
            if one_shot_hint:  # 찾았으면
                break  # 모드 루프 종료

    for date_mode in date_modes:  # 날짜모드 반복
        for tol in sorted(TOLS, reverse=True):  # ✅ tol 큰 값부터(한방 후보 확률↑)
            moves, status = build_moves_fast(
                idx=idx,
                pricebook=pricebook,
                df_submit=df_submit,
                submit_avail=submit_avail,
                yearly_min_stock_map=yearly_min_stock_map,
                target_code=target_code,
                target_name=target_name,
                target_use=target_use,
                target_date=target_date,
                need_qty=need_qty,
                tol=tol,
                date_mode=date_mode,
                allow_partial=allow_partial,
            )
            if len(moves) == 0:
                continue

            if not subs_ok_after_fast(idx, moves, target_date):
                continue

            moved_sum = int(sum(int(m["이동량"]) for m in moves))
            if moved_sum <= 0:
                continue

            if not allow_partial:
                if not is_solved_after_fast(idx, target_code, target_date, moved_sum):
                    continue

            idx.apply_adjustment_inplace(target_code, target_date, moves)

            tag = f"{int(tol*100)}%/{date_mode}"
            if one_shot_hint:  # 한방 힌트면
                tag += "/ONE_SHOT_HINT"  # 표시
            if status == "PARTIAL":
                tag += "/PARTIAL"
            tag += f"/{run_tag}"
            return True, moves, tag

    return False, [], ""

def apply_step0_1_split_outbound(
    df: pd.DataFrame,
    problem_items: list,
    worst_threshold: float = STEP01_WORST_THRESHOLD,
    max_future_months: int = STEP01_MAX_FUTURE_MONTHS,
    min_big_sale_ratio: float = STEP01_MIN_BIG_SALE_RATIO,
    exclude_party_keywords: tuple = STEP01_EXCLUDE_PARTY_KEYWORDS,
    verbose: bool = True,
):
    df = df.copy()

    # 적요 컬럼 보장
    if "적요" not in df.columns:
        df["적요"] = ""

    # 내부 고유키(원본행 재사용 방지용)
    if "_uid" not in df.columns:
        df["_uid"] = np.arange(len(df), dtype=int)

    # idx는 df 기준으로 생성
    df = df.reset_index(drop=True)
    idx = FastIndex(df)

    used_uids = set()
    logs = []

    def _ok_party(p: str) -> bool:
        p = safe_str(p)
        if "[이동]" in p:
            return False
        for kw in exclude_party_keywords:
            if kw and kw in p:
                return False
        return True

    if verbose:
        print(f"[STEP0-1] 대상 품목: {len(problem_items)} / worst_threshold={worst_threshold} / max_future_months={max_future_months}")

    for k, code in enumerate(problem_items, start=1):
        if verbose and (k % PRINT_EVERY_N_ITEMS == 0):
            print(f"[STEP0-1] {k}/{len(problem_items)} 진행중...")

        # 최신 idx/df 기준 item_df 구성
        item_mask = df[ITEM_COL].astype(str) == str(code)
        item_df = df[item_mask].copy()
        if len(item_df) == 0:
            continue

        # idx stock 덮어서 run 계산
        item_df[STOCK_COL] = idx.stock[item_df.index.to_numpy(dtype=int)]
        runs = build_runs_by_row_with_day_close_check(item_df)
        if not runs:
            continue

        # threshold 필터
        runs = [r for r in runs if float(r["worst_min"]) <= float(worst_threshold)]
        if not runs:
            continue

        # run은 시작일 빠른 순
        runs = sorted(runs, key=lambda x: x["run_start"])

        for run in runs:
            run_start = pd.Timestamp(run["run_start"])
            need = int(run["need_qty"])
            if need <= 0:
                continue

            # 후보 출고 전표:
            # - run_start 이전
            # - 출고수량 >= need * min_big_sale_ratio
            cand = df[
                (df[ITEM_COL].astype(str) == str(code)) &
                (pd.to_datetime(df[DATE_COL], errors="coerce") < run_start) &
                (pd.to_numeric(df[OUT_QTY_COL], errors="coerce").fillna(0).astype(int) >= int(math.ceil(need * float(min_big_sale_ratio))))
            ].copy()

            if len(cand) == 0:
                continue

            cand = cand[cand[PARTY_COL].apply(_ok_party)].copy()
            if len(cand) == 0:
                continue

            # 최신일 우선
            cand = cand.sort_values([DATE_COL], ascending=False, kind="mergesort")

            picked = None
            for _, r in cand.iterrows():
                uid = int(r["_uid"])
                if uid in used_uids:
                    continue

                base_date = pd.Timestamp(r[DATE_COL])
                party = safe_str(r[PARTY_COL])
                qty0 = pd.to_numeric(r[OUT_QTY_COL], errors="coerce")  # 출고수량 숫자변환
                qty0 = 0 if pd.isna(qty0) else int(qty0)  # NaN이면 0, 아니면 int

                split_qty = need
                remain_qty = qty0 - split_qty

                # "절반 이상 남아있는" 조건: remain_qty >= need
                if remain_qty < need:
                    continue

                # 미래 안전 날짜 탐색: (run_start+1) ~ (run_start + max_future_months)
                # 조건: 그 날짜 이후 최소재고에서 split_qty 빼도 음수 안되게
                future_ok_date = None
                start_d = run_start + pd.Timedelta(days=1)
                end_d = run_start + pd.DateOffset(months=int(max_future_months))

                for d in pd.date_range(start_d, end_d, freq="D"):
                    smin = idx.min_stock_after(str(code), d)
                    if (not pd.isna(smin)) and (float(smin) - float(split_qty) >= 0):
                        future_ok_date = pd.Timestamp(d)
                        break

                if future_ok_date is None:
                    continue

                picked = (r, base_date, party, qty0, split_qty, future_ok_date)
                break

            if picked is None:
                continue

            r, base_date, party, qty0, split_qty, future_ok_date = picked
            uid = int(r["_uid"])
            mmdd = base_date.strftime("%m.%d")

            # ----------------------------
            # 1) 원본 행: 수량은 그대로 (요구사항)
            # 2) 바로 아래: -split_qty 행 삽입 (출고 감소)
            # 3) 미래일자: +split_qty 행 추가 (예정 출고)
            # ----------------------------

            # (2) -split 행: 적요="출고 예정"
            neg_row = r.copy()
            neg_row[OUT_QTY_COL] = -int(split_qty)
            neg_row[OUT_PRICE_COL] = 0
            neg_row["적요"] = "출고 예정"
            neg_row["_uid"] = int(df["_uid"].max()) + 1  # 새 uid

            # (3) 미래 +split 행: 적요="mm.dd 미출고건"
            pos_row = r.copy()
            pos_row[DATE_COL] = future_ok_date
            pos_row[OUT_QTY_COL] = int(split_qty)
            pos_row[OUT_PRICE_COL] = 0
            pos_row["적요"] = f"{mmdd} 미출고건"
            pos_row["_uid"] = int(df["_uid"].max()) + 2  # 새 uid

            # 삽입 위치: 원본 행 바로 아래
            insert_pos = int(r.name) + 1
            df = pd.concat(
                [df.iloc[:insert_pos], pd.DataFrame([neg_row]), df.iloc[insert_pos:]],
                ignore_index=True
            )
            # 미래행은 맨 아래에 추가
            df = pd.concat([df, pd.DataFrame([pos_row])], ignore_index=True)

            # DF/IDX 재생성(행 삽입으로 인덱스가 바뀌었기 때문)
            df = df.reset_index(drop=True)
            idx = FastIndex(df)

            # 재고 반영:
            # - base_date에서 -(-split) = +split 효과
            idx.apply_plus_after(str(code), base_date, int(split_qty))
            # - future_ok_date에서 -(+split) = -split 효과
            idx.apply_minus_after(str(code), future_ok_date, int(split_qty))

            # df에도 최신 재고 반영(최소 안정화)
            df[STOCK_COL] = idx.stock

            used_uids.add(uid)

            logs.append({
                "단계": "STEP0-1",
                "문제일자": run_start,
                "문제품목코드": str(code),
                "거래처명": party,
                "기준출고일": base_date,
                "분리수량": int(split_qty),
                "감소행_적요": "출고 예정",
                "예정출고일": future_ok_date,
                "예정행_적요": f"{mmdd} 미출고건",
                "태그": f"STEP0-1_SPLIT_OUT(split={split_qty})",
            })


    return df.reset_index(drop=True), pd.DataFrame(logs)

def recalc_stock_item_with_anchor(g: pd.DataFrame, in_qty_col: str) -> pd.DataFrame:
    """
    재고 재계산:
      running += in_qty - out_qty
    [조정] 행은 앵커로 쓰되, stock이 0/빈값이면 앵커로 쓰지 않고 일반 delta 적용.
    """
    g = g.copy()
    inq = pd.to_numeric(g[in_qty_col], errors="coerce").fillna(0).astype(float).to_numpy()
    outq = pd.to_numeric(g[OUT_QTY_COL], errors="coerce").fillna(0).astype(float).to_numpy()
    stocks_src = pd.to_numeric(g[STOCK_COL], errors="coerce").fillna(0).astype(float).to_numpy()

    party = g[PARTY_COL].astype(str).fillna("").str.strip().to_numpy(dtype=object)
    is_adj = (party == ADJ_PARTY_VALUE)

    if len(stocks_src) == 0:
        return g

    stocks_new = stocks_src.copy()
    running = float(stocks_src[0])
    stocks_new[0] = running

    for i in range(1, len(stocks_new)):
        if is_adj[i] and (not np.isnan(stocks_src[i])) and (float(stocks_src[i]) != 0.0):
            running = float(stocks_src[i])
        else:
            running = running + float(inq[i]) - float(outq[i])
        stocks_new[i] = running

    g[STOCK_COL] = stocks_new
    return g

def reorder_intraday_by_stock_hint(g_day: pd.DataFrame, in_qty_col: str, prev_close_stock: Optional[float]):
    """
    같은 품목/같은 날짜 1일치(g_day)를 순서 재배치.
    - prev_close_stock: 직전 날짜의 마감 재고 (없으면 None)
    - 가능한 경우: current + (in-out) == row_stock 을 만족하는 행을 greedy로 이어 붙임.
    - 못 맞추면: _orig_order 기준으로 유지.
    """
    g = g_day.copy()
    g["_in"] = pd.to_numeric(g.get(in_qty_col, 0), errors="coerce").fillna(0).astype(float)
    g["_out"] = pd.to_numeric(g.get(OUT_QTY_COL, 0), errors="coerce").fillna(0).astype(float)
    g["_stk"] = pd.to_numeric(g.get(STOCK_COL, 0), errors="coerce").fillna(0).astype(float)

    # 기준 재고(당일 시작)
    if prev_close_stock is None:
        # prev 없으면, 첫 행의 stock을 시작점으로 둠(가장 보수적)
        current = float(g["_stk"].iloc[0]) if len(g) else 0.0
    else:
        current = float(prev_close_stock)

    remaining = g.sort_values(["_orig_order"], kind="mergesort").copy()
    ordered = []
    tol = 1e-6

    while len(remaining) > 0:
        # current + delta == stock 인 행을 우선 탐색
        expected = current + (remaining["_in"] - remaining["_out"]).to_numpy(dtype=float)
        diff = np.abs(expected - remaining["_stk"].to_numpy(dtype=float))
        hit = np.where(diff <= tol)[0]
        if len(hit) > 0:
            pick_i = int(hit[0])  # _orig_order 정렬 상태라서 첫 hit가 안정적
            row = remaining.iloc[pick_i:pick_i+1].copy()
            ordered.append(row)
            current = float(row["_stk"].iloc[0])  # stock 힌트를 신뢰
            remaining = pd.concat([remaining.iloc[:pick_i], remaining.iloc[pick_i+1:]], axis=0)
            continue

        # 매칭 실패: _orig_order 첫 행을 그대로 사용(최소 변경)
        row = remaining.iloc[:1].copy()
        # 이 경우엔 stock 힌트를 신뢰하지 말고 expected로 진행(후단 재계산에서 정합화)
        current = current + float(row["_in"].iloc[0]) - float(row["_out"].iloc[0])
        ordered.append(row)
        remaining = remaining.iloc[1:].copy()

    out = pd.concat(ordered, ignore_index=True)
    out = out.drop(columns=["_in", "_out", "_stk"], errors="ignore")
    return out

def apply_step00_pull_inbound_with_bonded(
    df_main: pd.DataFrame,
    df_bonded: pd.DataFrame,
    idx_main: FastIndex,
    problem_items: list,
    in_qty_col_main: str,
    in_qty_col_bonded: str,
    worst_threshold: float,
    allow_cross_month: bool,
    max_pull_days: int,
    max_pull_months: int,
    verbose: bool = True,
):
    df = df_main.copy()
    db = df_bonded.copy()

    # parse/normalize
    for xdf, incol in [(df, in_qty_col_main), (db, in_qty_col_bonded)]:
        ensure_cols(xdf, [DATE_COL, ITEM_COL, PARTY_COL, STOCK_COL, OUT_QTY_COL], ctx="[STEP00]")
        xdf[DATE_COL] = pd.to_datetime(xdf[DATE_COL], errors="coerce", format="mixed")
        xdf = xdf.dropna(subset=[DATE_COL]).copy()
        xdf[ITEM_COL] = xdf[ITEM_COL].astype(str).apply(normalize_item_code)
        xdf[incol] = pd.to_numeric(xdf[incol], errors="coerce").fillna(0).astype(float)
        xdf[OUT_QTY_COL] = pd.to_numeric(xdf[OUT_QTY_COL], errors="coerce").fillna(0).astype(float)
        xdf[STOCK_COL] = pd.to_numeric(xdf[STOCK_COL], errors="coerce").fillna(0).astype(float)
        xdf[PARTY_COL] = xdf[PARTY_COL].astype(str).fillna("")
        if xdf is df:
            df = xdf
        else:
            db = xdf

    df["_orig_order"] = np.arange(len(df), dtype=int)
    df["_step00_pri"] = 0
    db["_orig_order"] = np.arange(len(db), dtype=int)
    db["_step00_pri"] = 0

    def month_diff(a: pd.Timestamp, b: pd.Timestamp) -> int:
        return (b.year - a.year) * 12 + (b.month - a.month)

    def pick_ref_purchase(db_item: pd.DataFrame, move_old_date: pd.Timestamp):
        cand = db_item[(db_item[DATE_COL] <= move_old_date) & (db_item[in_qty_col_bonded] > 0)].copy()
        if len(cand) == 0:
            return None
        cand = cand.sort_values([DATE_COL, "_orig_order"], ascending=[False, False], kind="mergesort")
        return cand.iloc[0]

    def pick_bonded_move_row(db_item: pd.DataFrame, move_old_date: pd.Timestamp, qty_in_main: float):
        cand = db_item[
            (db_item[DATE_COL] == move_old_date)
            & (db_item[PARTY_COL].astype(str).apply(is_bonded_to_hq_move))
            & (pd.to_numeric(db_item[OUT_QTY_COL], errors="coerce").fillna(0) > 0)
        ].copy()
        if len(cand) == 0:
            return None
        cand["_out"] = pd.to_numeric(cand[OUT_QTY_COL], errors="coerce").fillna(0).astype(float)
        cand["ok"] = cand["_out"] >= float(qty_in_main)
        cand = cand.sort_values(["ok", "_out"], ascending=[False, True], kind="mergesort")
        return cand.iloc[0]

    logs_main = []
    logs_bonded = []
    affected_main = set()
    affected_bonded = set()
    used_main_vouchers = set()

    seen_main_log = set()
    seen_bonded_log = set()

    def _log_key_main(code, party, qty_in, old_date, new_date):
        return (str(code), safe_str(party), float(qty_in), pd.Timestamp(old_date), pd.Timestamp(new_date))

    def _log_key_bonded(code, buy_old, buy_qty, buy_new, mv_old, mv_out, mv_new, main_old, main_in, main_new):
        return (
            str(code),
            pd.Timestamp(buy_old), float(buy_qty), pd.Timestamp(buy_new),
            pd.Timestamp(mv_old), float(mv_out), pd.Timestamp(mv_new),
            pd.Timestamp(main_old), float(main_in), pd.Timestamp(main_new),
        )

    if verbose:
        print(
            f"[STEP00] items={len(problem_items)} worst<={worst_threshold} "
            f"cross_month={allow_cross_month} max_days={max_pull_days} max_months={max_pull_months} "
            f"resolve_ratio_min={STEP00_RESOLVE_RATIO_MIN} block_non_bonded_moves={STEP00_BLOCK_NON_BONDED_MOVES}"
        )

    for k, code in enumerate(problem_items, start=1):
        if verbose and (k % PRINT_EVERY_N_ITEMS == 0):
            print(f"[STEP00] {k}/{len(problem_items)} 진행중...")

        it = df[df[ITEM_COL].astype(str) == str(code)].copy()
        if len(it) == 0:
            continue

        it[STOCK_COL] = idx_main.stock[it.index.to_numpy(dtype=int)]
        runs = build_runs_by_row_with_day_close_check(it)
        runs = [r for r in runs if float(r["worst_min"]) <= float(worst_threshold)]
        runs = filter_runs_small_skip(runs, it, in_qty_col_main)
        if not runs:
            continue

        runs = sorted(runs, key=lambda x: x["run_start"])

        for run in runs:
            run_start = pd.Timestamp(run["run_start"])
            limit = run_start + pd.Timedelta(days=int(max_pull_days))

            cand = df[
                (df[ITEM_COL].astype(str) == str(code))
                & (df[DATE_COL] > run_start)
                & (df[DATE_COL] <= limit)
                & (df[in_qty_col_main] > 0)
            ].copy()
            if len(cand) == 0:
                continue

            if not allow_cross_month:
                cand = cand[
                    (cand[DATE_COL].dt.year == run_start.year)
                    & (cand[DATE_COL].dt.month == run_start.month)
                ].copy()
                if len(cand) == 0:
                    continue
            else:
                md = cand[DATE_COL].apply(lambda d: month_diff(run_start, pd.Timestamp(d)))
                cand = cand[md <= int(max_pull_months)].copy()
                if len(cand) == 0:
                    continue

            if STEP00_BLOCK_NON_BONDED_MOVES:
                def _ok_party(p):
                    p = safe_str(p)
                    if "[이동]" not in p:
                        return True
                    return is_bonded_to_hq_move(p)
                cand = cand[cand[PARTY_COL].apply(_ok_party)].copy()
                if len(cand) == 0:
                    continue

            cand = cand.sort_values([DATE_COL, "_orig_order"], kind="mergesort")
            r0 = cand.iloc[0]
            ridx_main = int(r0.name)

            if ridx_main in used_main_vouchers:
                continue

            old_date = pd.Timestamp(r0[DATE_COL])
            qty_in = float(r0[in_qty_col_main])
            party_main = str(r0[PARTY_COL]).strip()

            delta_days = int((old_date - run_start).days)
            if delta_days <= 0 or delta_days > int(max_pull_days):
                continue

            worst_need = abs(float(run["worst_min"]))
            if qty_in <= 0:
                continue
            resolve_ratio = min(qty_in, worst_need) / worst_need
            if resolve_ratio < float(STEP00_RESOLVE_RATIO_MIN):
                continue

            bonded_applied = "N"

            if STEP00_USE_BONDED_RULE and is_bonded_to_hq_move(party_main):
                db_item = db[db[ITEM_COL].astype(str) == str(code)].copy()
                if len(db_item) == 0:
                    continue

                mv = pick_bonded_move_row(db_item, move_old_date=old_date, qty_in_main=qty_in)
                if mv is None:
                    continue

                ridx_bonded_move = int(mv.name)
                bonded_move_out = to_float0(mv.get(OUT_QTY_COL, 0))

                ref = pick_ref_purchase(db_item, move_old_date=old_date)
                if ref is None:
                    continue

                ridx_bonded_buy = int(ref.name)
                buy_date = pd.Timestamp(ref[DATE_COL])
                buy_qty = float(ref[in_qty_col_bonded])

                if qty_in > buy_qty + 1e-9:
                    continue

                if (run_start.year, run_start.month) < (buy_date.year, buy_date.month):
                    continue

                buy_new_date = buy_date
                if run_start < buy_date:
                    same_month = (run_start.year == buy_date.year) and (run_start.month == buy_date.month)
                    if not same_month:
                        continue
                    db.loc[ridx_bonded_buy, DATE_COL] = run_start
                    db.loc[ridx_bonded_buy, "_step00_pri"] = -2
                    buy_new_date = run_start
                    affected_bonded.add(str(code))

                db.loc[ridx_bonded_move, DATE_COL] = run_start
                db.loc[ridx_bonded_move, "_step00_pri"] = -1
                affected_bonded.add(str(code))

                bonded_applied = "Y"

                keyB = _log_key_bonded(
                    code,
                    buy_date, buy_qty, buy_new_date,
                    old_date, bonded_move_out, run_start,
                    old_date, qty_in, run_start
                )
                if keyB not in seen_bonded_log:
                    seen_bonded_log.add(keyB)
                    logs_bonded.append(
                        {
                            "단계": "STEP00_BONDED",
                            "태그": "STEP00_BONDED_PULL_MOVE(+BUY_IF_SAME_MONTH)",
                            "품목코드": str(code),
                            "품목명": safe_str(db_item[NAME_COL].iloc[-1]) if NAME_COL in db_item.columns else "",
                            "구매전표_기존날짜": buy_date,
                            "구매전표_입고수량": buy_qty,
                            "구매전표_변경날짜": buy_new_date,
                            "이동전표_기존날짜": old_date,
                            "이동전표_출고수량": bonded_move_out,
                            "이동전표_변경날짜": run_start,
                            "본사입고전표_기존날짜": old_date,
                            "본사입고전표_입고수량": qty_in,
                            "본사입고전표_변경날짜": run_start,
                        }
                    )

            # 본사 전표 당김
            df.loc[ridx_main, DATE_COL] = run_start
            df.loc[ridx_main, "_step00_pri"] = -1
            affected_main.add(str(code))
            used_main_vouchers.add(ridx_main)

            tag = "STEP00_MAIN_PULL_INBOUND"
            if is_bonded_to_hq_move(party_main):
                tag += "/BONDED_MOVE"
            tag += f"/ratio={resolve_ratio:.2f}"

            keyM = _log_key_main(code, party_main, qty_in, old_date, run_start)
            if keyM not in seen_main_log:
                seen_main_log.add(keyM)
                logs_main.append(
                    {
                        "단계": "STEP00_MAIN",
                        "태그": tag,
                        "품목코드": str(code),
                        "거래처명": party_main,
                        "입고수량": qty_in,
                        "기존날짜": old_date,
                        "변경날짜": run_start,
                        "당김일수": delta_days,
                        "보세룰적용": bonded_applied,
                    }
                )

    # 재고 재계산: affected만
    if affected_main:
        others = df[~df[ITEM_COL].astype(str).isin(affected_main)].copy()
        parts = []
        for code in affected_main:
            g = df[df[ITEM_COL].astype(str) == str(code)].copy()
            g = g.sort_values([DATE_COL, "_step00_pri", "_orig_order"], kind="mergesort").reset_index(drop=True)
            g = recalc_stock_item_with_anchor(g, in_qty_col_main)
            parts.append(g)
        df = pd.concat([others] + parts, ignore_index=True)

    if affected_bonded:
        others = db[~db[ITEM_COL].astype(str).isin(affected_bonded)].copy()
        parts = []
        for code in affected_bonded:
            g = db[db[ITEM_COL].astype(str) == str(code)].copy()
            g = g.sort_values([DATE_COL, "_step00_pri", "_orig_order"], kind="mergesort").reset_index(drop=True)
            g = recalc_stock_item_with_anchor(g, in_qty_col_bonded)
            parts.append(g)
        db = pd.concat([others] + parts, ignore_index=True)

    df = df.drop(columns=["_orig_order", "_step00_pri"], errors="ignore")
    db = db.drop(columns=["_orig_order", "_step00_pri"], errors="ignore")

    df_log_main = pd.DataFrame(logs_main)
    if len(df_log_main) > 0:
        df_log_main = df_log_main.drop_duplicates(
            subset=["품목코드", "거래처명", "입고수량", "기존날짜", "변경날짜"],
            keep="first"
        ).reset_index(drop=True)

    df_log_bonded = pd.DataFrame(logs_bonded)
    if len(df_log_bonded) > 0:
        df_log_bonded = df_log_bonded.drop_duplicates(
            subset=["품목코드", "구매전표_기존날짜", "구매전표_입고수량", "구매전표_변경날짜",
                    "이동전표_기존날짜", "이동전표_출고수량", "이동전표_변경날짜",
                    "본사입고전표_기존날짜", "본사입고전표_입고수량", "본사입고전표_변경날짜"],
            keep="first"
        ).reset_index(drop=True)

    return df, db, df_log_main, df_log_bonded

def build_promo_pool_same_item_current(df_adj: pd.DataFrame, code: str) -> pd.DataFrame:
    d = df_adj[df_adj[ITEM_COL].astype(str) == str(code)].copy()
    d["_qty"] = pd.to_numeric(d[OUT_QTY_COL], errors="coerce").fillna(0).astype(int)
    d["_price"] = pd.to_numeric(d[OUT_PRICE_COL], errors="coerce").fillna(0).astype(float)
    d[PARTY_COL] = d[PARTY_COL].astype(str).fillna("")

    paid = (d["_qty"] > 0) & (d["_price"] > 0) & (d["_qty"] >= PROMO_PAID_MIN_QTY)
    free = (d["_qty"] > 0) & (d["_price"] == 0)

    d["_paid"] = paid
    d["_free"] = free

    cand_idx = []
    for (dt, party), g in d.groupby([DATE_COL, PARTY_COL], sort=False):
        if g["_paid"].any() and g["_free"].any():
            cand_idx.extend(g[g["_free"]].index.tolist())

    pool = df_adj.loc[cand_idx, [ITEM_COL, NAME_COL, DATE_COL, PARTY_COL, OUT_QTY_COL, OUT_PRICE_COL]].copy()
    pool = pool.rename(columns={OUT_QTY_COL: "무료출고수량"})
    pool["무료출고수량"] = pd.to_numeric(pool["무료출고수량"], errors="coerce").fillna(0).astype(int)
    pool = pool[pool["무료출고수량"] > 0].sort_values([DATE_COL, PARTY_COL], kind="mergesort").reset_index()
    return pool

def apply_step4_same_item_promo(  # STEP4 함수
    df_adj: pd.DataFrame,  # 수불부(작업본)
    idx: FastIndex,  # 빠른 인덱스
    problem_items: list,  # 문제 품목 리스트
    submit_map: dict,  # 제출본 맵
    real_map: dict,  # 실재고 맵
    sys_map: dict,  # 전산재고 맵
    avail_map: dict,  # 가용 맵
    verbose=True,  # 출력 여부
):
    df_adj = df_adj.copy()  # 원본 보호
    logs = []  # 로그 리스트

    if verbose:  # 출력이면
        print(f"[STEP4] 대상 품목: {len(problem_items)} (PROMO_PAID_MIN_QTY={PROMO_PAID_MIN_QTY})")  # 안내 출력

    for k, code in enumerate(problem_items, start=1):  # 품목 반복
        if verbose and (k % PRINT_EVERY_N_ITEMS == 0):  # 출력 주기면
            print(f"[STEP4] {k}/{len(problem_items)} 진행중...")  # 진행 출력

        item_df = df_adj[df_adj[ITEM_COL].astype(str) == str(code)].copy()  # 해당 품목 DF
        if len(item_df) == 0:  # 없으면
            continue  # 다음 품목

        item_df[STOCK_COL] = idx.stock[item_df.index.to_numpy(dtype=int)]  # 재고 덮기
        runs = build_runs_by_row_with_day_close_check(item_df)  # run 생성
        if len(runs) == 0:  # run 없으면
            continue  # 다음 품목

        sr = submit_map.get(str(code), None)  # 제출본 행
        item_name = str(sr.get(SUBMIT_NAME_COL, "")) if sr else safe_str(item_df[NAME_COL].iloc[-1])  # 품목명

        # =========================  # 구분선
        # ✅ STEP4 허용 + 상한 조건  # 섹션명
        # - SUBMIT_DIFF_COL(실재-전산) > 0 인 품목만 허용  # 조건
        # - STEP4에서 옮기는 총량은 diff까지만  # 상한
        # =========================  # 구분선
        submit_diff = None  # diff 기본값
        if sr is not None:  # 제출본 행 있으면
            submit_diff = to_float0(sr.get(SUBMIT_DIFF_COL, 0))  # diff 읽기
        else:  # 제출본 행 없으면
            submit_diff = float(real_map.get(str(code), 0.0)) - float(sys_map.get(str(code), 0.0))  # diff 계산

        diff_cap = int(max(0, np.floor(float(submit_diff))))  # diff 상한(정수)
        if diff_cap <= 0:  # 0 이하면
            continue  # ✅ STEP4 스킵

        moved_total_for_item = 0  # ✅ 품목별 STEP4 누적 이동량

        for run in runs:  # run 반복
            if moved_total_for_item >= diff_cap:  # 상한 다 썼으면
                break  # ✅ 더 이상 STEP4 금지

            run_start = pd.Timestamp(run["run_start"])  # run 시작일
            need_total = int(run["need_qty"])  # 필요 수량
            if need_total <= 0:  # 필요 없으면
                continue  # 다음 run

            remain_cap = diff_cap - moved_total_for_item  # ✅ 남은 diff 상한
            if remain_cap <= 0:  # 남은 상한 없으면
                break  # 종료

            need = int(min(need_total, remain_cap))  # ✅ need를 diff 상한으로 컷
            if need <= 0:  # 0이면
                continue  # 다음 run

            pool = build_promo_pool_same_item_current(df_adj, code)  # 무료출고 풀 생성
            if len(pool) == 0:  # 풀 없으면
                continue  # 다음 run

            problem_party = pick_party(item_df, run_start)  # 문제 거래처

            usable = pool[pool[DATE_COL] <= run_start].sort_values(  # run_start 이전만
                [DATE_COL, PARTY_COL], ascending=[False, False], kind="mergesort"  # 최신 우선
            ).copy()  # 복사
            if len(usable) == 0:  # 사용 가능 없으면
                continue  # 다음 run

            while need > 0 and len(usable) > 0:  # 필요 남고 후보 있으면
                r = usable.iloc[0]  # 첫 후보
                promo_idx = int(r["index"])  # 원본 인덱스
                promo_date = pd.Timestamp(r[DATE_COL])  # 전표일자
                promo_party = str(r[PARTY_COL])  # 거래처
                free_before = int(r["무료출고수량"])  # 기존 무료출고

                use_qty = int(min(need, free_before))  # 사용할 수량
                free_after = free_before - use_qty  # 사용 후 무료출고
                if use_qty <= 0:  # 0이면
                    break  # 종료

                df_adj.loc[promo_idx, OUT_QTY_COL] = float(free_after)  # 무료출고 감소 반영
                idx.apply_plus_after(str(code), promo_date, use_qty)  # 재고 + 반영
                apply_sys_delta(sys_map, real_map, avail_map, str(code), +use_qty)  # 전산 + 반영

                need -= use_qty  # 필요 감소
                moved_total_for_item += use_qty  # ✅ 누적 이동 증가

                logs.append(  # 로그 기록
                    {
                        "단계": "STEP4",  # 단계
                        "문제일자": run_start,  # 문제일
                        "문제품목코드": str(code),  # 품목코드
                        "문제품목명": item_name,  # 품목명
                        "문제거래처명": problem_party,  # 문제거래처
                        "문제일_재고(참고)": float(run["worst_min"]),  # 참고재고
                        "필요이동량": int(need_total),  # 필요량(원need)
                        "대체품목_거래처명": promo_party,  # 거래처
                        "대체품목_날짜": promo_date,  # 전표일
                        "이동량": int(use_qty),  # 이동량
                        "태그": "STEP4_PROMO_FREE_OUT_REDUCE",  # 태그
                    }
                )

                if moved_total_for_item >= diff_cap:  # ✅ 상한 다 썼으면
                    break  # while 종료

                usable = usable.iloc[1:].reset_index(drop=True)  # 다음 후보로 이동

    return df_adj, pd.DataFrame(logs)  # 결과 반환

def apply_step5_reprice_sales(
    df_adj: pd.DataFrame,
    idx: FastIndex,
    problem_items: list,
    submit_map: dict,
    real_map: dict,
    sys_map: dict,
    avail_map: dict,
    verbose=True,
):
    df_adj = df_adj.copy()
    logs = []
    used_step5_rows = set()

    if verbose:
        print(f"[STEP5] 대상 품목: {len(problem_items)} (min_qty={STEP5_MIN_QTY}, max_price_diff={STEP5_MAX_PRICE_DIFF})")

    for k, code in enumerate(problem_items, start=1):
        if verbose and (k % PRINT_EVERY_N_ITEMS == 0):
            print(f"[STEP5] {k}/{len(problem_items)} 진행중...")

        item_df0 = df_adj[df_adj[ITEM_COL].astype(str) == str(code)].copy()
        if len(item_df0) == 0:
            continue

        item_df0[STOCK_COL] = idx.stock[item_df0.index.to_numpy(dtype=int)]
        runs = build_runs_by_row_with_day_close_check(item_df0)
        if len(runs) == 0:
            continue

        sr = submit_map.get(str(code), None)
        item_name = str(sr.get(SUBMIT_NAME_COL, "")) if sr else safe_str(item_df0[NAME_COL].iloc[-1])

        for run in runs:
            run_start = pd.Timestamp(run["run_start"])
            worst_min = float(run["worst_min"])
            need_total = int(run["need_qty"])
            if need_total <= 0:
                continue

            required_recover = 0
            if worst_min <= float(STEP5_RUN_MIN):
                required_recover = int(math.ceil(float(STEP5_RUN_MIN_RECOVER_RATIO) * float(need_total)))

            base = df_adj[df_adj[ITEM_COL].astype(str) == str(code)].copy()
            base["_idx"] = base.index.astype(int)
            base["_qty"] = pd.to_numeric(base[OUT_QTY_COL], errors="coerce").fillna(0).astype(int)
            base["_price"] = pd.to_numeric(base[OUT_PRICE_COL], errors="coerce").fillna(0).astype(float)
            base[PARTY_COL] = base[PARTY_COL].astype(str).fillna("")

            usable = base[
                (pd.to_datetime(base[DATE_COL], errors="coerce") <= run_start)
                & (base["_qty"] >= STEP5_MIN_QTY)
                & (base["_price"] > 0)
            ].copy()
            usable = usable.sort_values([DATE_COL], ascending=False, kind="mergesort")
            if len(usable) == 0:
                continue

            item_df_run = df_adj[df_adj[ITEM_COL].astype(str) == str(code)].copy()
            item_df_run[STOCK_COL] = idx.stock[item_df_run.index.to_numpy(dtype=int)]
            problem_party = pick_party(item_df_run, run_start)

            # 1) 계획(Plan)
            plan = []
            remaining = int(need_total)

            for _, r in usable.iterrows():
                if remaining <= 0:
                    break

                ridx = int(r["_idx"])
                if ridx in used_step5_rows:
                    continue

                dt = pd.Timestamp(r[DATE_COL])
                party = safe_str(r.get(PARTY_COL, ""))

                q0 = int(r["_qty"])
                p0 = float(r["_price"])
                if q0 < STEP5_MIN_QTY or p0 <= 0:
                    continue

                total = int(round(q0 * p0))
                max_reduce = min(remaining, q0 - 1)
                if max_reduce <= 0:
                    continue

                best = None
                for dec in range(max_reduce, 0, -1):
                    q1 = q0 - dec
                    if q1 <= 0:
                        continue
                    if total % q1 != 0:
                        continue
                    p1 = total // q1
                    if p1 <= 0:
                        continue
                    if abs(p1 - p0) / p0 > STEP5_MAX_PRICE_DIFF:
                        continue
                    best = (dec, q1, p1)
                    break

                if best is None:
                    continue

                dec, q1, p1 = best
                plan.append((ridx, dt, party, q0, p0, q1, p1, dec))
                remaining -= dec

            plan_recover = int(sum(x[-1] for x in plan))

            # run 회복률 조건(큰 마이너스 run만)
            if required_recover > 0 and plan_recover < required_recover:
                continue

            # 2) 적용(Apply)
            need = int(need_total)
            for (ridx, dt, party, q0, p0, q1, p1, dec) in plan:
                if need <= 0:
                    break

                df_adj.loc[ridx, OUT_QTY_COL] = float(q1)
                df_adj.loc[ridx, OUT_PRICE_COL] = float(p1)
                used_step5_rows.add(ridx)

                idx.apply_plus_after(str(code), dt, dec)
                apply_sys_delta(sys_map, real_map, avail_map, str(code), +dec)

                need -= dec

                logs.append(
                    {
                        "단계": "STEP5",
                        "문제일자": run_start,
                        "문제품목코드": str(code),
                        "문제품목명": item_name,
                        "문제거래처명": problem_party,
                        "문제일_재고(참고)": worst_min,
                        "필요이동량": need_total,
                        "대체품목_거래처명": party,
                        "대체품목_날짜": dt,
                        "이동량": dec,
                        "태그": f"STEP5_REPRICE(q:{q0}->{q1}, p:{int(p0)}->{int(p1)})/runRecover={plan_recover}/{required_recover}",
                        "전표idx": ridx,
                    }
                )

    return df_adj, pd.DataFrame(logs)

def get_problem_items(df: pd.DataFrame, idx: FastIndex) -> list:
    all_items = df[ITEM_COL].dropna().astype(str).unique().tolist()
    problem = []
    for code in all_items:
        it = df[df[ITEM_COL].astype(str) == str(code)].copy()
        if len(it) == 0:
            continue
        it[STOCK_COL] = idx.stock[it.index.to_numpy(dtype=int)]
        if len(build_runs_by_row_with_day_close_check(it)) > 0:
            problem.append(str(code))
    return problem

def count_resolved(before_list: list, after_list: list) -> int:
    return len(set(before_list) - set(after_list))

def worst_minus_by_item(df: pd.DataFrame) -> dict:
    out = {}
    for code, g in df.groupby(ITEM_COL, sort=False):
        runs = build_runs_by_row_with_day_close_check(g)
        if len(runs) == 0:
            out[str(code)] = 0.0
        else:
            out[str(code)] = float(min(r["worst_min"] for r in runs))
    return out

def build_summary_sheet(df_before: pd.DataFrame, df_after: pd.DataFrame, stage_name: str) -> pd.DataFrame:
    b = worst_minus_by_item(df_before)
    a = worst_minus_by_item(df_after)

    codes = sorted(set(b.keys()) | set(a.keys()))
    rows = []
    for code in codes:
        wb = float(b.get(code, 0.0))
        wa = float(a.get(code, 0.0))
        rows.append(
            {
                "구분": stage_name,
                "문제품목코드": code,
                "최고마이너스_보정전": wb,
                "최고마이너스_보정후": wa,
                "개선량(+면개선)": wa - wb,
                "해결여부": "해결" if wa >= 0 else "미해결",
            }
        )
    return pd.DataFrame(rows)

def build_step_counts(df_move: pd.DataFrame) -> pd.DataFrame:
    """df_move(이력이력_STEP0to5)에서 단계별 행 개수 + 전체합계"""
    if df_move is None or len(df_move) == 0 or ("단계" not in df_move.columns):
        return pd.DataFrame([{"단계": "TOTAL", "행개수": 0}])

    vc = df_move["단계"].astype(str).value_counts(dropna=False).reset_index()
    vc.columns = ["단계", "행개수"]
    total = int(vc["행개수"].sum())
    vc = pd.concat([vc, pd.DataFrame([{"단계": "TOTAL", "행개수": total}])], ignore_index=True)

    order = {
        "STEP0": 0,
        "STEP0-1": 1,
        "STEP00_MAIN": 2,
        "STEP00_BONDED": 3,
        "STEP1": 4, "STEP2": 5, "STEP3": 6, "STEP4": 7, "STEP5": 8,
    }
    vc["_ord"] = vc["단계"].map(lambda x: order.get(x, 999))
    vc = vc.sort_values(["_ord", "단계"], kind="mergesort").drop(columns=["_ord"]).reset_index(drop=True)
    return vc

def count_remaining_problem_stats(df_final: pd.DataFrame) -> pd.DataFrame:
    """
    STEP5 이후에도 중간마이너스(run)가 남아있는:
    - 품목수
    - RUN개수
    - 행개수(각 run 길이 합)
    """
    if df_final is None or len(df_final) == 0:
        return pd.DataFrame([
            {"구분": "STEP5_이후_잔여_품목수", "값": 0},
            {"구분": "STEP5_이후_잔여_RUN개수", "값": 0},
            {"구분": "STEP5_이후_잔여_행개수", "값": 0},
        ])

    idx = FastIndex(df_final)
    problem_items = get_problem_items(df_final, idx)

    run_cnt = 0
    row_cnt = 0

    for code in problem_items:
        item_df = df_final[df_final[ITEM_COL].astype(str) == str(code)].copy()
        if len(item_df) == 0:
            continue
        item_df[STOCK_COL] = idx.stock[item_df.index.to_numpy(dtype=int)]
        runs = build_runs_by_row_with_day_close_check(item_df)
        run_cnt += len(runs)
        for r in runs:
            row_cnt += int(r["run_end_row"] - r["run_start_row"] + 1)

    return pd.DataFrame([
        {"구분": "STEP5_이후_잔여_품목수", "값": int(len(problem_items))},
        {"구분": "STEP5_이후_잔여_RUN개수", "값": int(run_cnt)},
        {"구분": "STEP5_이후_잔여_행개수", "값": int(row_cnt)},
    ])

def solve_all_with_step0_to5_with_step00_bonded(
    df_suful: pd.DataFrame, df_submit: pd.DataFrame, df_bonded: pd.DataFrame,
    out_dir=None, ts=None,
):
    """out_dir, ts 가 주어지면 STEP0~3 직후 1회, STEP0~5(최종) 완료 시 1회 저장합니다."""
    t0 = time.time()

    # 제출본 상태
    df_submit_template, submit_map, real_map, sys_map, avail_map = init_submit_state(df_submit)
    df_submit2 = df_submit_template.copy()
    df_submit2[ITEM_COL] = df_submit2[ITEM_COL].astype(str).apply(normalize_item_code)
    df_submit2[SUBMIT_USE_COL] = df_submit2[SUBMIT_USE_COL].astype(str).str.strip()
    df_submit2[SUBMIT_NAME_COL] = df_submit2[SUBMIT_NAME_COL].astype(str)

    # 본사 수불부 전처리
    df = df_suful.copy()
    ensure_cols(df, [DATE_COL, ITEM_COL, STOCK_COL, WH_COL, PARTY_COL, NAME_COL, OUT_QTY_COL, OUT_PRICE_COL], ctx="[수불부]")

    df[DATE_COL] = df[DATE_COL].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce", format="mixed")
    df = df.dropna(subset=[DATE_COL]).copy()

    df[ITEM_COL] = df[ITEM_COL].astype(str).apply(normalize_item_code)
    df[STOCK_COL] = pd.to_numeric(df[STOCK_COL], errors="coerce").fillna(0).astype(float)
    df[OUT_QTY_COL] = pd.to_numeric(df[OUT_QTY_COL], errors="coerce").fillna(0).astype(float)
    df[OUT_PRICE_COL] = pd.to_numeric(df[OUT_PRICE_COL], errors="coerce").fillna(0).astype(float)
    df[PARTY_COL] = df[PARTY_COL].astype(str).fillna("")
    df[WH_COL] = df[WH_COL].astype(str).fillna("")
    df[NAME_COL] = df[NAME_COL].astype(str).fillna("")
    if UNIT_COL not in df.columns:
        df[UNIT_COL] = "EA"
    if "적요" not in df.columns:
        df["적요"] = ""

    # 내부 고유키
    df["_uid"] = np.arange(len(df), dtype=int)

    in_qty_col = detect_in_qty_col(df)

    # 전처리1: 중복 제거
    df, dup_report = remove_yearly_duplicate_blocks_strict_v2(df, min_block_len=10)

    # 본사만
    df = df[df[WH_COL].astype(str).str.contains(WAREHOUSE_KEYWORD, na=False)].copy()

    # 전처리2: 비대상 제거
    df, exclude_report = remove_non_product_items(df)

    # # 전처리3: 중복 제거 (일자+품목코드+거래처+입출고방향)
    # if DEDUP_SUFUL_DUPLICATES:

    #     df, dedup_report = dedup_by_date_item_party_direction(df, in_qty_col)




    # 연중 최소 재고 (참고용)
    yearly_min_stock_map, yearly_min_stock_df = build_yearly_min_stock_table(df)
    
    df = reorder_within_same_day_by_stock_chain(df, in_qty_col)

    df_preprocessed_base = df.copy().reset_index(drop=True)

    # START 카운트
    idx_pre = FastIndex(df_preprocessed_base)
    all_items_pre = df_preprocessed_base[ITEM_COL].dropna().astype(str).unique().tolist()
    problem_pre = get_problem_items(df_preprocessed_base, idx_pre)
    print("========================================")
    print(f"[START] 전체 품목 수: {len(all_items_pre)} / 문제품목 수: {len(problem_pre)}")
    print("========================================")

    # STEP0
    print("========================================")
    print(f"[STEP0] 시작 - 문제품목 수: {len(problem_pre)}")

    df0 = df_preprocessed_base.copy()
    idx0 = FastIndex(df0)
    year_guess = int(pd.Timestamp(df0[DATE_COL].max()).year) if len(df0) else datetime.now().year

    df0_mod, df_step0_log = apply_step0_adjustments(
        df_pre=df0,
        idx=idx0,
        year_for_jan=year_guess,
        real_map=real_map,
        sys_map=sys_map,
        avail_map=avail_map,
        step0_start_month=1,
        step0_end_month=6,   # ✅ 6월까지 확대
        verbose=True,
    )
    df0_mod[STOCK_COL] = idx0.stock
    df0_mod = df0_mod.reset_index(drop=True)
    # ✅ STEP0 후: 동일 일자 내 정렬/재고 재계산
    df0_mod = reorder_and_recalc_inventory(df0_mod, in_qty_col)

    idx0_chk = FastIndex(df0_mod)
    problem_after0 = get_problem_items(df0_mod, idx0_chk)
    print(f"[STEP0] 종료 - 남은 문제품목 수: {len(problem_after0)} / 해결된 품목 수: {count_resolved(problem_pre, problem_after0)}")
    print("========================================")

    df_submit_step0 = materialize_submit_df(df_submit_template, real_map, sys_map)

    # STEP0-1
    print("========================================")
    print(f"[STEP0-1] 시작 - 문제품목 수: {len(problem_after0)}")

    df01_mod, df_step0_1_log = apply_step0_1_split_outbound(
        df=df0_mod,
        problem_items=problem_after0,
        worst_threshold=STEP01_WORST_THRESHOLD,
        max_future_months=STEP01_MAX_FUTURE_MONTHS,
        min_big_sale_ratio=STEP01_MIN_BIG_SALE_RATIO,
        exclude_party_keywords=STEP01_EXCLUDE_PARTY_KEYWORDS,
        verbose=True,
    )
    df01_mod = df01_mod.reset_index(drop=True)
    # ✅ STEP0-1 후: 동일 일자 내 정렬/재고 재계산
    df01_mod = reorder_and_recalc_inventory(df01_mod, in_qty_col)
    idx01_chk = FastIndex(df01_mod)
    problem_after01 = get_problem_items(df01_mod, idx01_chk)
    print(f"[STEP0-1] 종료 - 남은 문제품목 수: {len(problem_after01)} / 해결된 품목 수: {count_resolved(problem_after0, problem_after01)}")
    print("========================================")

    # STEP00
    print("========================================")
    in_qty_col_bonded = detect_in_qty_col(df_bonded)
    print(f"[STEP00] 시작 - 문제품목 수: {len(problem_after01)} / 본사입고={in_qty_col} / 보세입고={in_qty_col_bonded}")

    df00 = df01_mod.copy().reset_index(drop=True)
    idx00 = FastIndex(df00)

    df00_mod, df_bonded_mod, df_step00_main_log, df_step00_bonded_log = apply_step00_pull_inbound_with_bonded(
        df_main=df00,
        df_bonded=df_bonded,
        idx_main=idx00,
        problem_items=problem_after01,
        in_qty_col_main=in_qty_col,
        in_qty_col_bonded=in_qty_col_bonded,
        worst_threshold=STEP00_WORST_THRESHOLD,
        allow_cross_month=STEP00_ALLOW_CROSS_MONTH,
        max_pull_days=STEP00_MAX_PULL_DAYS,
        max_pull_months=STEP00_MAX_PULL_MONTHS,
        verbose=True,
    )

    df00_mod = df00_mod.reset_index(drop=True)
    # ✅ STEP00 후: 동일 일자 내 정렬/재고 재계산
    df00_mod = reorder_and_recalc_inventory(df00_mod, in_qty_col)
    idx00_chk = FastIndex(df00_mod)
    problem_after00 = get_problem_items(df00_mod, idx00_chk)
    print(f"[STEP00] 종료 - 남은 문제품목 수: {len(problem_after00)} / 해결된 품목 수: {count_resolved(problem_after01, problem_after00)}")
    print("========================================")

    df_submit_step00 = materialize_submit_df(df_submit_template, real_map, sys_map)

    # STEP1~5 준비 (조정전표 제거)
    df_work = df00_mod[df00_mod[PARTY_COL].astype(str).str.strip() != ADJ_PARTY_VALUE].copy().reset_index(drop=True)
    idx = FastIndex(df_work)
    pricebook = PriceBook(df_work)

    # STEP1 시작
    problem_items = get_problem_items(df_work, idx)
    print("========================================")
    print(f"[STEP1] 시작 - 문제품목 수: {len(problem_items)}")
    print("========================================")

    move_logs = []

    # STEP1
    for n, code in enumerate(problem_items, start=1):
        if n % PRINT_EVERY_N_ITEMS == 0:
            print(f"[STEP1] {n}/{len(problem_items)} elapsed={int(time.time() - t0)}s")

        sr = submit_map.get(str(code), None)
        target_name = str(sr.get(SUBMIT_NAME_COL, "")) if sr else ""
        target_use = str(sr.get(SUBMIT_USE_COL, "")).strip() if sr else ""
        while True:
            item_df = df_work[df_work[ITEM_COL].astype(str) == str(code)].copy()
            if len(item_df) == 0:
                break
            if sr is None:
                target_name = safe_str(item_df[NAME_COL].iloc[-1])
            item_df[STOCK_COL] = idx.stock[item_df.index.to_numpy(dtype=int)]
            runs_raw = build_runs_by_row_with_day_close_check(item_df)
            runs_raw = sorted(runs_raw, key=lambda x: x["run_start"])
            if len(runs_raw) == 0:
                break
            best_run = runs_raw[0]
            target_date = pd.Timestamp(best_run["run_start"])
            need_qty = int(best_run["need_qty"])
            problem_party = pick_party(item_df, target_date)
            ok1, moves1, tag1 = try_fix_once_fast(
                idx=idx,
                pricebook=pricebook,
                df_submit=df_submit2,
                submit_avail=avail_map,
                yearly_min_stock_map=yearly_min_stock_map,
                target_code=code,
                target_name=target_name,
                target_use=target_use,
                target_date=target_date,
                need_qty=need_qty,
                run_tag="RUN1(반복,FULL,first-neg)",
                allow_partial=False,
                force_date_modes=["M"],
            )
            if not ok1:
                break
            moved_sum = int(sum(int(m["이동량"]) for m in moves1))
            apply_sys_delta(sys_map, real_map, avail_map, str(code), +moved_sum)
            for m in moves1:
                sub = str(m["대체품목코드"])
                used = int(m["이동량"])
                before_av = int(avail_map.get(sub, 0))
                apply_sys_delta(sys_map, real_map, avail_map, sub, -used)
                after_av = int(avail_map.get(sub, 0))
                move_logs.append({
                    "단계": "STEP1",
                    "문제일자": target_date,
                    "문제품목코드": str(code),
                    "문제품목명": target_name,
                    "문제거래처명": problem_party,
                    "문제일_재고(참고)": float(best_run.get("worst_min", 0)),
                    "필요이동량": need_qty,
                    "대체품목코드": sub,
                    "대체품목명": m.get("대체품목명", ""),
                    "대체품목_거래처명": m.get("대체품목_거래처명", ""),
                    "대체품목_날짜": target_date,
                    "이동량": used,
                    "태그": tag1,
                    "대체품목_pool_전(전산-실재)": before_av,
                    "대체품목_pool_후(전산-실재)": after_av,
                })

    idx1_chk = FastIndex(df_work)
    df_work[STOCK_COL] = idx.stock  # 재고 동기화
    df_work = df_work.reset_index(drop=True)  # 인덱스 초기화
    idx = FastIndex(df_work)  # idx 재생성
    problem_after1 = get_problem_items(df_work, idx1_chk)
    print("========================================")
    print(f"[STEP1] 종료 - 남은 문제품목 수: {len(problem_after1)} / 해결된 품목 수: {count_resolved(problem_items, problem_after1)}")
    print("========================================")

    # =========================  # 구분선
    # STEP2~STEP3  # 섹션명
    # =========================  # 구분선
    remain_items = problem_after1  # 남은 문제품목
    df_work[STOCK_COL] = idx.stock  # 재고 동기화
    df_work = df_work.reset_index(drop=True)  # 인덱스 초기화
    idx = FastIndex(df_work)  # idx 재생성
    pricebook = PriceBook(df_work)  # pricebook 재생성

    print("========================================")  # 구분선
    print(f"[STEP2~STEP3] 시작 - 문제품목 수: {len(remain_items)}")  # 시작 출력
    print("========================================")  # 구분선

    t_global = time.time()  # STEP2~3 시작시간

    for i, code in enumerate(remain_items, start=1):  # 품목 반복
        if i % PRINT_EVERY_N_ITEMS == 0:  # 출력 주기
            print(f"[STEP2~STEP3] {i}/{len(remain_items)} elapsed={int(time.time() - t_global)}s")  # 진행 출력

        t_item_start = time.time()  # 품목 시작시간
        attempts = 0  # 시도 횟수

        item_df_base = df_work[df_work[ITEM_COL].astype(str) == str(code)].copy()  # 품목 DF
        if len(item_df_base) == 0:  # 없으면
            continue  # 다음 품목

        sr = submit_map.get(str(code), None)  # 제출본 행
        target_name = str(sr.get(SUBMIT_NAME_COL, "")) if sr else safe_str(item_df_base[NAME_COL].iloc[-1])  # 품목명
        target_use = str(sr.get(SUBMIT_USE_COL, "")).strip() if sr else ""  # 실-전

        # =========================  # 구분선
        # STEP2 (M-only, FULL)  # 섹션명
        # =========================  # 구분선
        while True:  # 반복 시도
            if time.time() - t_item_start > MAX_ITEM_SECONDS:  # 시간 초과
                break  # 종료
            if attempts >= MAX_ITEM_ATTEMPTS:  # 시도 초과
                break  # 종료

            item_df = df_work[df_work[ITEM_COL].astype(str) == str(code)].copy()  # 최신 품목 DF
            if len(item_df) == 0:  # 없으면
                break  # 종료

            item_df[STOCK_COL] = idx.stock[item_df.index.to_numpy(dtype=int)]  # 재고 덮기
            runs_raw = build_runs_by_row_with_day_close_check(item_df)  # run 생성
            runs_raw = sorted(runs_raw, key=lambda x: x["run_start"])  # 기본 정렬(안정)

            # =========================  # 구분선
            # ✅ "단일 후보 FULL 가능 run" 우선 정렬  # 설명
            # - 빠른 체크(one_shot_full_possible_fast)로 True인 run을 앞으로  # 규칙
            # - 그 다음은 need 큰 run 우선(한번에 많이 해결)  # 규칙
            # =========================  # 구분선
            runs_scored = []  # 점수 run 리스트
            for r in runs_raw:  # run 반복
                td = pd.Timestamp(r["run_start"])  # 날짜
                need0 = int(r["need_qty"])  # need
                one = one_shot_full_possible_fast(  # ✅ 빠른 한방 체크(M-only 기준)
                    idx=idx,  # idx
                    pricebook=pricebook,  # pricebook
                    df_submit=df_submit2,  # 제출본
                    submit_avail=avail_map,  # 가용
                    target_code=code,  # 대상코드
                    target_name=target_name,  # 대상명
                    target_use=target_use,  # 실-전
                    target_date=td,  # 날짜
                    need_qty=need0,  # need
                    tol=max(TOLS),  # ✅ 가장 큰 tol로(한방 후보 놓치지 않게)
                    date_mode="M",  # M-only
                )
                rr = dict(r)  # run 복사
                rr["_one_shot"] = 1 if one else 0  # 점수 저장
                runs_scored.append(rr)  # 추가

            runs = sorted(  # 최종 정렬
                runs_scored,  # scored
                key=lambda x: (x["_one_shot"], x["need_qty"]),  # one_shot 우선, need 큰순
                reverse=True,  # 내림차순
            )
            if len(runs) == 0:  # run 없으면
                break  # 종료

            progressed = False  # 진행 여부

            for run in runs:  # run 반복
                if time.time() - t_item_start > MAX_ITEM_SECONDS:  # 시간 초과
                    break  # 종료
                if attempts >= MAX_ITEM_ATTEMPTS:  # 시도 초과
                    break  # 종료

                target_date = pd.Timestamp(run.get("first_neg_date", run["run_start"]))  # ✅ 최초 음수 날짜로 통일
                need_qty = int(run["need_qty"])  # 최악 기준 need 유지
                problem_party = pick_party(item_df, target_date)  # 거래처

                attempts += 1  # 시도 증가
                ok2, moves2, tag2 = try_fix_once_fast(  # 1회 시도
                    idx=idx,  # idx
                    pricebook=pricebook,  # pricebook
                    df_submit=df_submit2,  # 제출본
                    submit_avail=avail_map,  # 가용
                    yearly_min_stock_map=yearly_min_stock_map,  # 참고
                    target_code=code,  # 대상코드
                    target_name=target_name,  # 대상명
                    target_use=target_use,  # 실-전
                    target_date=target_date,  # 날짜
                    need_qty=need_qty,  # 필요
                    run_tag="RUN2(M-only,row-run)",  # 태그
                    allow_partial=False,  # FULL
                    force_date_modes=["M"],  # M-only
                )

                if not ok2:  # 실패면
                    continue  # 다음 run

                progressed = True  # 진행됨
                moved_sum = int(sum(int(m["이동량"]) for m in moves2))  # 이동합
                apply_sys_delta(sys_map, real_map, avail_map, str(code), +moved_sum)  # 대상 증가

                for m in moves2:  # move 기록
                    sub = str(m["대체품목코드"])  # 대체코드
                    used = int(m["이동량"])  # 이동량

                    before_av = int(avail_map.get(sub, 0))  # 전 가용
                    apply_sys_delta(sys_map, real_map, avail_map, sub, -used)  # 대체 감소
                    after_av = int(avail_map.get(sub, 0))  # 후 가용

                    move_logs.append(  # 로그 추가
                        {
                            "단계": "STEP2",  # 단계
                            "문제일자": target_date,  # 문제일
                            "문제품목코드": str(code),  # 문제코드
                            "문제품목명": target_name,  # 문제명
                            "문제거래처명": problem_party,  # 거래처
                            "문제일_재고(참고)": float(run["worst_min"]),  # 참고재고
                            "필요이동량": need_qty,  # 필요
                            "대체품목코드": sub,  # 대체코드
                            "대체품목명": m.get("대체품목명", ""),  # 대체명
                            "대체품목_거래처명": m.get("대체품목_거래처명", ""),  # 대체거래처
                            "대체품목_날짜": m.get("대체품목_날짜", pd.NaT),  # 대체일자
                            "이동량": used,  # 이동량
                            "태그": tag2,  # 태그
                            "대체품목_pool_전(전산-실재)": before_av,  # 전
                            "대체품목_pool_후(전산-실재)": after_av,  # 후
                        }
                    )

                break  # run 1개 성공하면 다시 runs 재계산하려고 탈출

            if not progressed:  # 진행 없으면
                break  # 종료

        # =========================  # 구분선
        # STEP3 (M-only, PARTIAL)  # 섹션명
        # =========================  # 구분선
        item_df = df_work[df_work[ITEM_COL].astype(str) == str(code)].copy()  # 최신 품목 DF
        if len(item_df) == 0:  # 없으면
            continue  # 다음 품목

        item_df[STOCK_COL] = idx.stock[item_df.index.to_numpy(dtype=int)]  # 재고 덮기
        runs_raw = build_runs_by_row_with_day_close_check(item_df)  # run 생성
        runs_raw = sorted(runs_raw, key=lambda x: x["run_start"])  # 기본 정렬(안정)

        # =========================  # 구분선
        # ✅ "단일 후보 FULL 가능 run" 우선 정렬  # 설명
        # - 빠른 체크(one_shot_full_possible_fast)로 True인 run을 앞으로  # 규칙
        # - 그 다음은 need 큰 run 우선(한번에 많이 해결)  # 규칙
        # =========================  # 구분선
        runs_scored = []  # 점수 run 리스트
        for r in runs_raw:  # run 반복
            td = pd.Timestamp(r["run_start"])  # 날짜
            need0 = int(r["need_qty"])  # need
            one = one_shot_full_possible_fast(  # ✅ 빠른 한방 체크(M-only 기준)
                idx=idx,  # idx
                pricebook=pricebook,  # pricebook
                df_submit=df_submit2,  # 제출본
                submit_avail=avail_map,  # 가용
                target_code=code,  # 대상코드
                target_name=target_name,  # 대상명
                target_use=target_use,  # 실-전
                target_date=td,  # 날짜
                need_qty=need0,  # need
                tol=max(TOLS),  # ✅ 가장 큰 tol로(한방 후보 놓치지 않게)
                date_mode="M",  # M-only
            )
            rr = dict(r)  # run 복사
            rr["_one_shot"] = 1 if one else 0  # 점수 저장
            runs_scored.append(rr)  # 추가

        runs = sorted(  # 최종 정렬
            runs_scored,  # scored
            key=lambda x: (x["_one_shot"], x["need_qty"]),  # one_shot 우선, need 큰순
            reverse=True,  # 내림차순
        )
        if len(runs) == 0:  # run 없으면
            continue  # 다음 품목

        for run in runs:  # run 반복
            if time.time() - t_item_start > MAX_ITEM_SECONDS:  # 시간 초과
                break  # 종료
            if attempts >= MAX_ITEM_ATTEMPTS:  # 시도 초과
                break  # 종료

            target_date = pd.Timestamp(run.get("first_neg_date", run["run_start"]))  # ✅ 최초 음수 날짜로 통일
            need_qty = int(run["need_qty"])  # 최악 기준 need 유지
            problem_party = pick_party(item_df, target_date)  # 거래처

            attempts += 1  # 시도 증가
            ok3, moves3, tag3 = try_fix_once_fast(  # 1회 시도
                idx=idx,  # idx
                pricebook=pricebook,  # pricebook
                df_submit=df_submit2,  # 제출본
                submit_avail=avail_map,  # 가용
                yearly_min_stock_map=yearly_min_stock_map,  # 참고
                target_code=code,  # 대상코드
                target_name=target_name,  # 대상명
                target_use=target_use,  # 실-전
                target_date=target_date,  # 날짜
                need_qty=need_qty,  # 필요
                run_tag="RUN3(PARTIAL,M-only,row-run)",  # 태그
                allow_partial=True,  # PARTIAL
                force_date_modes=["M"],  # M-only
            )

            if not ok3:  # 실패면
                continue  # 다음 run

            moved_sum = int(sum(int(m["이동량"]) for m in moves3))  # 이동합
            apply_sys_delta(sys_map, real_map, avail_map, str(code), +moved_sum)  # 대상 증가

            for m in moves3:  # move 기록
                sub = str(m["대체품목코드"])  # 대체코드
                used = int(m["이동량"])  # 이동량

                before_av = int(avail_map.get(sub, 0))  # 전 가용
                apply_sys_delta(sys_map, real_map, avail_map, sub, -used)  # 대체 감소
                after_av = int(avail_map.get(sub, 0))  # 후 가용

                move_logs.append(  # 로그 추가
                    {
                        "단계": "STEP3",  # 단계
                        "문제일자": target_date,  # 문제일
                        "문제품목코드": str(code),  # 문제코드
                        "문제품목명": target_name,  # 문제명
                        "문제거래처명": problem_party,  # 거래처
                        "문제일_재고(참고)": float(run["worst_min"]),  # 참고재고
                        "필요이동량": need_qty,  # 필요
                        "대체품목코드": sub,  # 대체코드
                        "대체품목명": m.get("대체품목명", ""),  # 대체명
                        "대체품목_거래처명": m.get("대체품목_거래처명", ""),  # 대체거래처
                        "대체품목_날짜": m.get("대체품목_날짜", pd.NaT),  # 대체일자
                        "이동량": used,  # 이동량
                        "태그": tag3,  # 태그
                        "대체품목_pool_전(전산-실재)": before_av,  # 전
                        "대체품목_pool_후(전산-실재)": after_av,  # 후
                    }
                )

            break  # PARTIAL도 run 1개 성공하면 다음 품목으로

    # STEP2~STEP3 종료 동기화
    df_work[STOCK_COL] = idx.stock  # 재고 동기화
    df_work = df_work.reset_index(drop=True)  # 인덱스 초기화
    idx = FastIndex(df_work)  # idx 재생성
    pricebook = PriceBook(df_work)  # pricebook 재생성

    problem_after3 = get_problem_items(df_work, idx)  # 문제품목 재계산
    print("========================================")  # 구분선
    print(f"[STEP2~STEP3] 종료 - 남은 문제품목 수: {len(problem_after3)} / 해결된 품목 수: {count_resolved(remain_items, problem_after3)}")  # 종료 출력
    print("========================================")  # 구분선

    # STEP0~3 직후 저장 (STEP5 제외, 그 전까지 결과물)
    if out_dir and ts:
        df_submit_step03 = materialize_submit_df(df_submit_template, real_map, sys_map)
        df_move_03 = pd.DataFrame(move_logs)
        if len(df_step0_log) > 0:
            df_move_03 = pd.concat([df_step0_log, df_move_03], ignore_index=True) if len(df_move_03) else df_step0_log.copy()
        if len(df_step0_1_log) > 0:
            df_move_03 = pd.concat([df_move_03, df_step0_1_log], ignore_index=True)
        if len(df_step00_main_log) > 0:
            df_move_03 = pd.concat([df_move_03, df_step00_main_log], ignore_index=True)
        if len(df_step00_bonded_log) > 0:
            df_move_03 = pd.concat([df_move_03, df_step00_bonded_log], ignore_index=True)
        if len(df_move_03) > 0 and "문제일자" in df_move_03.columns:
            df_move_03 = df_move_03.sort_values(["문제일자"], kind="mergesort").reset_index(drop=True)
        df_step_counts_03 = build_step_counts(df_move_03)
        summary_03 = pd.concat([
            build_summary_sheet(df_preprocessed_base, df0_mod, "STEP0"),
            build_summary_sheet(df0_mod, df01_mod, "STEP0-1"),
            build_summary_sheet(df01_mod, df00_mod, "STEP00"),
            build_summary_sheet(
                df00_mod[df00_mod[PARTY_COL].astype(str).str.strip() != ADJ_PARTY_VALUE].copy().reset_index(drop=True),
                df_work,
                "STEP1~STEP3",
            ),
        ], ignore_index=True)
        df_remaining_03 = count_remaining_problem_stats(df_work)
        out_step03 = f"{out_dir}/STEP0~3_수불부_제출본_{ts}.xlsx"
        print("(STEP0~3) 저장 중...")
        with pd.ExcelWriter(out_step03, engine="openpyxl") as writer:
            df_work.to_excel(writer, sheet_name="수불부_STEP0~3", index=False)
            df_submit_step03.to_excel(writer, sheet_name="제출본_STEP0~3", index=False)
            df_move_03.to_excel(writer, sheet_name="이력이력_STEP0to3", index=False)
            df_step_counts_03.to_excel(writer, sheet_name="STEP별_이력_행개수", index=False)
            summary_03.to_excel(writer, sheet_name="요약", index=False)
            yearly_min_stock_df.to_excel(writer, sheet_name="연중최소재고", index=False)
            df_remaining_03.to_excel(writer, sheet_name="STEP3_이후_잔여(품목RUN행)", index=False)
        print(f"(STEP0~3) 저장 완료: {out_step03}")

    # =========================  # 구분선
    # STEP4  # 섹션명
    # =========================  # 구분선
    print("========================================")  # 구분선
    print(f"[STEP4] 시작 - 문제품목 수: {len(problem_after3)}")  # 시작 출력
    print("========================================")  # 구분선

    df_work[STOCK_COL] = idx.stock  # 재고 동기화
    df_work = df_work.reset_index(drop=True)  # 인덱스 초기화
    idx = FastIndex(df_work)  # idx 재생성
    pricebook = PriceBook(df_work)  # pricebook 재생성

    if len(problem_after3) > 0:  # 대상 있으면
        df_work, df_step4 = apply_step4_same_item_promo(  # STEP4 실행
            df_adj=df_work,  # DF
            idx=idx,  # idx
            problem_items=problem_after3,  # 대상
            submit_map=submit_map,  # 제출맵
            real_map=real_map,  # 실재
            sys_map=sys_map,  # 전산
            avail_map=avail_map,  # 가용
            verbose=True,  # 출력
        )
    else:  # 대상 없으면
        df_step4 = pd.DataFrame()  # 빈 DF

    df_work[STOCK_COL] = idx.stock  # 재고 동기화
    df_work = df_work.reset_index(drop=True)  # 인덱스 초기화
    idx = FastIndex(df_work)  # idx 재생성
    pricebook = PriceBook(df_work)  # pricebook 재생성

    problem_after4 = get_problem_items(df_work, idx)  # 문제품목 재계산
    print("========================================")  # 구분선
    print(f"[STEP4] 종료 - 남은 문제품목 수: {len(problem_after4)} / 해결된 품목 수: {count_resolved(problem_after3, problem_after4)}")  # 종료 출력
    print("========================================")  # 구분선


    # =========================  # 구분선
    # STEP5  # 섹션명
    # =========================  # 구분선
    print("========================================")  # 구분선
    print(f"[STEP5] 시작 - 문제품목 수: {len(problem_after4)}")  # 시작 출력
    print("========================================")  # 구분선

    df_work[STOCK_COL] = idx.stock  # 재고 동기화
    df_work = df_work.reset_index(drop=True)  # 인덱스 초기화
    idx = FastIndex(df_work)  # idx 재생성
    pricebook = PriceBook(df_work)  # pricebook 재생성

    if len(problem_after4) > 0:  # 대상 있으면
        df_work, df_step5 = apply_step5_reprice_sales(  # STEP5 실행
            df_adj=df_work,  # DF
            idx=idx,  # idx
            problem_items=problem_after4,  # 대상
            submit_map=submit_map,  # 제출맵
            real_map=real_map,  # 실재
            sys_map=sys_map,  # 전산
            avail_map=avail_map,  # 가용
            verbose=True,  # 출력
        )
    else:  # 대상 없으면
        df_step5 = pd.DataFrame()  # 빈 DF

    df_work[STOCK_COL] = idx.stock  # 재고 동기화
    df_work = df_work.reset_index(drop=True)  # 인덱스 초기화
    idx = FastIndex(df_work)  # idx 재생성
    pricebook = PriceBook(df_work)  # pricebook 재생성

    problem_after5 = get_problem_items(df_work, idx)  # 문제품목 재계산
    print("========================================")  # 구분선
    print(f"[STEP5] 종료 - 남은 문제품목 수: {len(problem_after5)} / 해결된 품목 수: {count_resolved(problem_after4, problem_after5)}")  # 종료 출력
    print("========================================")  # 구분선

    # 최종 동기화(제출본 생성 직전)
    df_work[STOCK_COL] = idx.stock  # 최종 재고 동기화

    # ✅ (NEW) 같은 날짜 내 입고/출고 흐름 맞게 재정렬 + 재고 재계산
    df_work = resort_within_day_and_recalc(df_work, in_qty_col)

    # (중요) 재정렬/재계산 후 idx도 재생성 (이후 저장/잔여분석 안정화)
    df_work = df_work.reset_index(drop=True)
    idx = FastIndex(df_work)

    df_submit_final = materialize_submit_df(df_submit_template, real_map, sys_map)

    # 로그 합치기
    df_move = pd.DataFrame(move_logs)

    if len(df_step0_log) > 0:
        df_move = pd.concat([df_step0_log, df_move], ignore_index=True) if len(df_move) else df_step0_log.copy()
    if len(df_step0_1_log) > 0:
        df_move = pd.concat([df_move, df_step0_1_log], ignore_index=True)
    if len(df_step00_main_log) > 0:
        df_move = pd.concat([df_move, df_step00_main_log], ignore_index=True)
    if len(df_step00_bonded_log) > 0:
        df_move = pd.concat([df_move, df_step00_bonded_log], ignore_index=True)
    if len(df_step4) > 0:
        df_move = pd.concat([df_move, df_step4], ignore_index=True)
    if len(df_step5) > 0:
        df_move = pd.concat([df_move, df_step5], ignore_index=True)

    if len(df_move) > 0 and "문제일자" in df_move.columns:
        df_move = df_move.sort_values(["문제일자"], kind="mergesort").reset_index(drop=True)

    df_step_counts = build_step_counts(df_move)

    # 이력이력 중복 방어
    dedup_cols = [c for c in ["단계","태그","문제품목코드","문제일자","대체품목코드","대체품목_날짜","이동량","기존날짜","변경날짜","거래처명","입고수량"] if c in df_move.columns]
    if len(dedup_cols) > 0:
        df_move = df_move.drop_duplicates(subset=dedup_cols, keep="first").reset_index(drop=True)

    summary0 = build_summary_sheet(df_preprocessed_base, df0_mod, "STEP0")
    summary01 = build_summary_sheet(df0_mod, df01_mod, "STEP0-1")
    summary00 = build_summary_sheet(df01_mod, df00_mod, "STEP00")
    summary5 = build_summary_sheet(
        df00_mod[df00_mod[PARTY_COL].astype(str).str.strip() != ADJ_PARTY_VALUE].copy().reset_index(drop=True),
        df_work,
        "STEP1~STEP5",
    )
    df_summary = pd.concat([summary0, summary01, summary00, summary5], ignore_index=True)

    preprocess_report = {
        "dup_report": dup_report,
        "exclude_report": exclude_report,
        "yearly_min_stock_df": yearly_min_stock_df,
        "step0_log": df_step0_log,
        "step0_1_log": df_step0_1_log,
        "step00_main_log": df_step00_main_log,
        "step00_bonded_log": df_step00_bonded_log,
        "df_preprocessed_base": df_preprocessed_base,
        "in_qty_col_main": in_qty_col,
        "in_qty_col_bonded": in_qty_col_bonded,
    }

    print("========================================")
    print("[DONE] 전체 파이프라인 종료")
    print(f" - START 문제품목: {len(problem_pre)}")
    print(f" - STEP0    이후: {len(problem_after0)}")
    print(f" - STEP0-1  이후: {len(problem_after01)}")
    print(f" - STEP00   이후: {len(problem_after00)}")
    print(f" - STEP1    이후: {len(problem_after1)}")
    print(f" - STEP3    이후: {len(problem_after3)}")
    print(f" - STEP4    이후: {len(problem_after4)}")
    print(f" - STEP5    이후: {len(problem_after5)}")
    print(f" - 총 소요(s): {int(time.time() - t0)}")
    print("========================================")

    df_remaining_after5 = count_remaining_problem_stats(df_work)

    # 최종(STEP0~5) 저장
    if out_dir and ts:
        out_final = f"{out_dir}/STEP0~5_수불부_제출본_{ts}.xlsx"
        print("(STEP0~5) 저장 중...")
        with pd.ExcelWriter(out_final, engine="openpyxl") as writer:
            df_work.to_excel(writer, sheet_name="수불부_최종(STEP5)", index=False)
            df_submit_final.to_excel(writer, sheet_name="제출본_최종(STEP5)", index=False)
            df_move.to_excel(writer, sheet_name="이력이력_STEP0to5", index=False)
            df_step_counts.to_excel(writer, sheet_name="STEP별_이력_행개수", index=False)
            df_summary.to_excel(writer, sheet_name="요약", index=False)
            yearly_min_stock_df.to_excel(writer, sheet_name="연중최소재고", index=False)
            df_remaining_after5.to_excel(writer, sheet_name="STEP5_이후_잔여(품목RUN행)", index=False)
        print(f"(STEP0~5) 저장 완료: {out_final}")

    return {
        "df_step0_suful": df0_mod,
        "df_submit_step0": df_submit_step0,
        "df_step01_suful": df01_mod,
        "df_step01_log": df_step0_1_log,
        "df_step00_suful": df00_mod,
        "df_submit_step00": df_submit_step00,
        "df_bonded_step00": df_bonded_mod,
        "df_step00_bonded_log": df_step00_bonded_log,
        "df_step00_main_log": df_step00_main_log,
        "df_final_suful": df_work,
        "df_submit_final": df_submit_final,
        "df_move": df_move,
        "df_summary": df_summary,
        "preprocess_report": preprocess_report,
        "df_step_counts": df_step_counts,
        "df_remaining_after5": df_remaining_after5,
    }


# =========================================================
# 추가 유틸: 반복 헤더 행 제거 (수불부/보세수불부 공통)
# =========================================================
def drop_repeated_header_rows(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    엑셀에서 헤더(컬럼명) 행이 중간에 반복되는 케이스 제거.
    - DATE_COL/ITEM_COL/STOCK_COL 등이 '일자', '품목코드', '재고수량' 같은 텍스트로 들어간 행 제거
    - 컬럼명이 그대로 데이터로 들어간 행 제거
    반환: (정리된 df, 제거된 행 report)
    """
    if df is None or len(df) == 0:
        return df, pd.DataFrame()

    d = df.copy()
    cols = list(d.columns)

    # 1) "컬럼명이 그대로 값으로 들어간 행" 제거
    def _count_colname_matches(row):
        cnt = 0
        for c in cols:
            if str(row.get(c, "")).strip() == str(c).strip():
                cnt += 1
        return cnt

    mask_colname_row = d.apply(lambda r: _count_colname_matches(r) >= max(2, len(cols)//3), axis=1)

    # 2) 대표 컬럼 기준 "헤더 텍스트" 제거
    def _is_header_like(v: str) -> bool:
        s = str(v).strip()
        return s in {
            DATE_COL, ITEM_COL, STOCK_COL, WH_COL, PARTY_COL, NAME_COL, OUT_QTY_COL, OUT_PRICE_COL,
            "일자", "품목코드", "재고수량", "창고", "거래처", "품목명"
        }

    masks = []
    for c in [DATE_COL, ITEM_COL, STOCK_COL, WH_COL, PARTY_COL, NAME_COL]:
        if c in d.columns:
            masks.append(d[c].astype(str).map(_is_header_like))
    mask_header_text = masks[0].copy() if masks else pd.Series(False, index=d.index)
    for m in masks[1:]:
        mask_header_text = mask_header_text | m

    mask = (mask_colname_row | mask_header_text).fillna(False)

    removed = d[mask].copy()
    kept = d[~mask].copy()

    rep = pd.DataFrame({
        "removed_rows": [int(mask.sum())],
        "total_rows": [int(len(d))],
        "removed_ratio": [float(mask.mean()) if len(d) else 0.0]
    })
    return kept, rep


# =========================================================
# STEP4용(=기존 STEP1~3) 함수 분리: 로직은 solve 내부 코드 그대로 사용
# =========================================================
def apply_step1_to3_combined(
    df_in: pd.DataFrame,
    submit_map: dict,
    df_submit2: pd.DataFrame,
    real_map: dict,
    sys_map: dict,
    avail_map: dict,
    yearly_min_stock_map: dict,
) -> tuple[pd.DataFrame, pd.DataFrame, list]:
    """
    기존 V16 solve의 'STEP1~3' 구간을 하나의 함수로 분리.
    입력 df_in은 STEP00 이후(=기존 STEP00 완료된 수불부)라고 가정하지만,
    안전을 위해 조정전표(ADJ_PARTY_VALUE)는 내부에서 제거합니다.
    반환:
      - df_after3 (STEP3까지 적용된 수불부)
      - df_move_log (이동 로그 DataFrame)
      - problem_after3 (남은 문제품목 리스트)
    """
    # STEP1~5 준비 (조정전표 제거)
    df_work = df_in[df_in[PARTY_COL].astype(str).str.strip() != ADJ_PARTY_VALUE].copy().reset_index(drop=True)
    idx = FastIndex(df_work)
    pricebook = PriceBook(df_work)

    # STEP1 시작
    problem_items = get_problem_items(df_work, idx)
    print("========================================")
    print(f"[STEP4-1] (기존 STEP1) 시작 - 문제품목 수: {len(problem_items)}")
    print("========================================")

    move_logs: list[dict] = []

    # (기존 STEP1) RUN1
    for n, code in enumerate(problem_items, start=1):
        if n % PRINT_EVERY_N_ITEMS == 0:
            print(f"[STEP4-1] {n}/{len(problem_items)}")

        sr = submit_map.get(str(code), None)
        target_name = str(sr.get(SUBMIT_NAME_COL, "")) if sr else ""
        target_use = str(sr.get(SUBMIT_USE_COL, "")).strip() if sr else ""
        while True:
            item_df = df_work[df_work[ITEM_COL].astype(str) == str(code)].copy()
            if len(item_df) == 0:
                break
            if sr is None:
                target_name = safe_str(item_df[NAME_COL].iloc[-1])
            item_df[STOCK_COL] = idx.stock[item_df.index.to_numpy(dtype=int)]
            runs_raw = build_runs_by_row_with_day_close_check(item_df)
            runs_raw = sorted(runs_raw, key=lambda x: x["run_start"])
            if len(runs_raw) == 0:
                break
            best_run = runs_raw[0]
            target_date = pd.Timestamp(best_run["run_start"])
            need_qty = int(best_run["need_qty"])
            problem_party = pick_party(item_df, target_date)
            ok1, moves1, tag1 = try_fix_once_fast(
                idx=idx,
                pricebook=pricebook,
                df_submit=df_submit2,
                submit_avail=avail_map,
                yearly_min_stock_map=yearly_min_stock_map,
                target_code=code,
                target_name=target_name,
                target_use=target_use,
                target_date=target_date,
                need_qty=need_qty,
                run_tag="RUN1(반복,FULL,first-neg)",
                allow_partial=False,
                force_date_modes=["M"],
            )
            if not ok1:
                break
            moved_sum = int(sum(int(m["이동량"]) for m in moves1))
            apply_sys_delta(sys_map, real_map, avail_map, str(code), +moved_sum)
            for m in moves1:
                sub = str(m["대체품목코드"])
                used = int(m["이동량"])
                before_av = int(avail_map.get(sub, 0))
                apply_sys_delta(sys_map, real_map, avail_map, sub, -used)
                after_av = int(avail_map.get(sub, 0))
                move_logs.append({
                    "단계": "STEP1",
                    "문제일자": target_date,
                    "문제품목코드": str(code),
                    "문제품목명": target_name,
                    "거래처": problem_party,
                    "필요수량": need_qty,
                    "대체품목코드": sub,
                    "대체품목명": str(m.get("대체품목명", "")),
                    "이동량": used,
                    "대체품목_가능재고_before": before_av,
                    "대체품목_가능재고_after": after_av,
                    "run_tag": tag1,
                })

    # STEP1 종료 재정리
    df_work[STOCK_COL] = idx.stock
    df_work = df_work.reset_index(drop=True)
    in_qty_col = detect_in_qty_col(df_work)
    df_work = reorder_and_recalc_inventory(df_work, in_qty_col)
    idx = FastIndex(df_work)
    pricebook = PriceBook(df_work)

    problem_after1 = get_problem_items(df_work, idx)
    print("========================================")
    print(f"[STEP4-1] (기존 STEP1) 종료 - 남은 문제품목 수: {len(problem_after1)}")
    print("========================================")

    # (기존 STEP2) RUN2
    print("========================================")
    print(f"[STEP4-2] (기존 STEP2) 시작 - 문제품목 수: {len(problem_after1)}")
    print("========================================")

    for n, code in enumerate(problem_after1, start=1):
        if n % PRINT_EVERY_N_ITEMS == 0:
            print(f"[STEP4-2] {n}/{len(problem_after1)}")

        sr = submit_map.get(str(code), None)
        target_name = str(sr.get(SUBMIT_NAME_COL, "")) if sr else ""
        target_use = str(sr.get(SUBMIT_USE_COL, "")).strip() if sr else ""

        item_df = df_work[df_work[ITEM_COL].astype(str) == str(code)].copy()
        if len(item_df) == 0:
            continue
        if sr is None:
            target_name = safe_str(item_df[NAME_COL].iloc[-1])

        item_df[STOCK_COL] = idx.stock[item_df.index.to_numpy(dtype=int)]
        runs_raw = build_runs_by_row_with_day_close_check(item_df)
        runs_raw = sorted(runs_raw, key=lambda x: x["run_start"])
        if len(runs_raw) == 0:
            continue
        best_run = runs_raw[0]
        target_date = pd.Timestamp(best_run["run_start"])
        need_qty = int(best_run["need_qty"])
        problem_party = pick_party(item_df, target_date)

        ok2, moves2, tag2 = try_fix_once_fast(
            idx=idx,
            pricebook=pricebook,
            df_submit=df_submit2,
            submit_avail=avail_map,
            yearly_min_stock_map=yearly_min_stock_map,
            target_code=code,
            target_name=target_name,
            target_use=target_use,
            target_date=target_date,
            need_qty=need_qty,
            run_tag="RUN2(단회,PARTIAL,first-neg)",
            allow_partial=True,
            force_date_modes=["M"],
        )
        if not ok2:
            continue
        moved_sum = int(sum(int(m["이동량"]) for m in moves2))
        apply_sys_delta(sys_map, real_map, avail_map, str(code), +moved_sum)
        for m in moves2:
            sub = str(m["대체품목코드"])
            used = int(m["이동량"])
            before_av = int(avail_map.get(sub, 0))
            apply_sys_delta(sys_map, real_map, avail_map, sub, -used)
            after_av = int(avail_map.get(sub, 0))
            move_logs.append({
                "단계": "STEP2",
                "문제일자": target_date,
                "문제품목코드": str(code),
                "문제품목명": target_name,
                "거래처": problem_party,
                "필요수량": need_qty,
                "대체품목코드": sub,
                "대체품목명": str(m.get("대체품목명", "")),
                "이동량": used,
                "대체품목_가능재고_before": before_av,
                "대체품목_가능재고_after": after_av,
                "run_tag": tag2,
            })

    # STEP2 종료 재정리
    df_work[STOCK_COL] = idx.stock
    df_work = df_work.reset_index(drop=True)
    df_work = reorder_and_recalc_inventory(df_work, in_qty_col)
    idx = FastIndex(df_work)
    pricebook = PriceBook(df_work)

    problem_after2 = get_problem_items(df_work, idx)
    print("========================================")
    print(f"[STEP4-2] (기존 STEP2) 종료 - 남은 문제품목 수: {len(problem_after2)}")
    print("========================================")

    # (기존 STEP3) RUN3
    print("========================================")
    print(f"[STEP4-3] (기존 STEP3) 시작 - 문제품목 수: {len(problem_after2)}")
    print("========================================")

    for n, code in enumerate(problem_after2, start=1):
        if n % PRINT_EVERY_N_ITEMS == 0:
            print(f"[STEP4-3] {n}/{len(problem_after2)}")

        sr = submit_map.get(str(code), None)
        target_name = str(sr.get(SUBMIT_NAME_COL, "")) if sr else ""
        target_use = str(sr.get(SUBMIT_USE_COL, "")).strip() if sr else ""

        item_df = df_work[df_work[ITEM_COL].astype(str) == str(code)].copy()
        if len(item_df) == 0:
            continue
        if sr is None:
            target_name = safe_str(item_df[NAME_COL].iloc[-1])

        item_df[STOCK_COL] = idx.stock[item_df.index.to_numpy(dtype=int)]
        runs_raw = build_runs_by_row_with_day_close_check(item_df)
        runs_raw = sorted(runs_raw, key=lambda x: x["run_start"])
        if len(runs_raw) == 0:
            continue
        best_run = runs_raw[0]
        target_date = pd.Timestamp(best_run["run_start"])
        need_qty = int(best_run["need_qty"])
        problem_party = pick_party(item_df, target_date)

        ok3, moves3, tag3 = try_fix_once_fast(
            idx=idx,
            pricebook=pricebook,
            df_submit=df_submit2,
            submit_avail=avail_map,
            yearly_min_stock_map=yearly_min_stock_map,
            target_code=code,
            target_name=target_name,
            target_use=target_use,
            target_date=target_date,
            need_qty=need_qty,
            run_tag="RUN3(단회,PARTIAL,last-neg)",
            allow_partial=True,
            force_date_modes=["L"],
        )
        if not ok3:
            continue
        moved_sum = int(sum(int(m["이동량"]) for m in moves3))
        apply_sys_delta(sys_map, real_map, avail_map, str(code), +moved_sum)
        for m in moves3:
            sub = str(m["대체품목코드"])
            used = int(m["이동량"])
            before_av = int(avail_map.get(sub, 0))
            apply_sys_delta(sys_map, real_map, avail_map, sub, -used)
            after_av = int(avail_map.get(sub, 0))
            move_logs.append({
                "단계": "STEP3",
                "문제일자": target_date,
                "문제품목코드": str(code),
                "문제품목명": target_name,
                "거래처": problem_party,
                "필요수량": need_qty,
                "대체품목코드": sub,
                "대체품목명": str(m.get("대체품목명", "")),
                "이동량": used,
                "대체품목_가능재고_before": before_av,
                "대체품목_가능재고_after": after_av,
                "run_tag": tag3,
            })

    # STEP3 종료 재정리
    df_work[STOCK_COL] = idx.stock
    df_work = df_work.reset_index(drop=True)
    df_work = reorder_and_recalc_inventory(df_work, in_qty_col)
    idx = FastIndex(df_work)

    problem_after3 = get_problem_items(df_work, idx)
    print("========================================")
    print(f"[STEP4-3] (기존 STEP3) 종료 - 남은 문제품목 수: {len(problem_after3)}")
    print("========================================")

    df_move_log = pd.DataFrame(move_logs)
    return df_work, df_move_log, problem_after3


In [5]:

# =========================
# 공통: 파일 선택 & 로드
# =========================
def _pick_three_files(step_title: str):
    root = Tk()
    root.attributes("-topmost", True)
    root.withdraw()
    try:
        root.focus_force()
    except Exception:
        pass

    suful_path = filedialog.askopenfilename(
        title=f"[{step_title}] 수불부 엑셀 선택 (이 STEP의 입력)",
        filetypes=[("Excel files", "*.xlsx *.xls")],
    )
    if not suful_path:
        root.destroy()
        raise SystemExit("수불부 선택 취소")

    submit_path = filedialog.askopenfilename(
        title=f"[{step_title}] 제출본(실-전) 엑셀 선택 (이 STEP의 입력)",
        filetypes=[("Excel files", "*.xlsx *.xls")],
    )
    if not submit_path:
        root.destroy()
        raise SystemExit("제출본 선택 취소")

    bonded_path = filedialog.askopenfilename(
        title=f"[{step_title}] 보세 수불부 엑셀 선택 (이 STEP의 입력)",
        filetypes=[("Excel files", "*.xlsx *.xls")],
    )
    if not bonded_path:
        root.destroy()
        raise SystemExit("보세 수불부 선택 취소")

    out_dir = filedialog.askdirectory(title=f"[{step_title}] 저장 폴더 선택")
    if not out_dir:
        root.destroy()
        raise SystemExit("저장 폴더 선택 취소")

    df_suful, sh1 = read_excel_best_sheet(suful_path, include_keywords=["수불부"])
    df_submit, sh2 = read_excel_best_sheet(submit_path, include_keywords=["제출본", "실-전"])
    df_bonded, sh3 = read_excel_best_sheet(bonded_path, include_keywords=["수불부"])

    print(f" - 수불부 시트: {sh1}")
    print(f" - 제출본 시트: {sh2}")
    print(f" - 보세수불부 시트: {sh3}")

    return df_suful, df_submit, df_bonded, out_dir


def _save_step_result(out_path: str, df_suful: pd.DataFrame, df_submit: pd.DataFrame, logs: dict):
    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        df_suful.to_excel(writer, sheet_name="수불부", index=False)
        df_submit.to_excel(writer, sheet_name="제출본", index=False)
        # logs: DataFrame 또는 dict/str 모두 처리
        for k, v in (logs or {}).items():
            if isinstance(v, pd.DataFrame):
                v.to_excel(writer, sheet_name=str(k)[:31], index=False)
            else:
                pd.DataFrame([{"value": str(v)}]).to_excel(writer, sheet_name=str(k)[:31], index=False)
    print(f"저장 완료: {out_path}")


## STEP 0 (원본 전처리)

In [ ]:

df_suful, df_submit, df_bonded, out_dir = _pick_three_files("STEP 0 (원본 전처리)")
ts = datetime.now().strftime("%Y%m%d_%H%M%S")

# ---------- 수불부 기본 정리 ----------
df = df_suful.copy()
ensure_cols(df, [DATE_COL, ITEM_COL, STOCK_COL, WH_COL, PARTY_COL, NAME_COL, OUT_QTY_COL, OUT_PRICE_COL], ctx="[수불부]")
df[DATE_COL] = df[DATE_COL].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce", format="mixed")
df = df.dropna(subset=[DATE_COL]).copy()

df[ITEM_COL] = df[ITEM_COL].astype(str).apply(normalize_item_code)
df[STOCK_COL] = pd.to_numeric(df[STOCK_COL], errors="coerce").fillna(0).astype(float)
df[OUT_QTY_COL] = pd.to_numeric(df[OUT_QTY_COL], errors="coerce").fillna(0).astype(float)
df[OUT_PRICE_COL] = pd.to_numeric(df[OUT_PRICE_COL], errors="coerce").fillna(0).astype(float)
df[PARTY_COL] = df[PARTY_COL].astype(str).fillna("")
df[WH_COL] = df[WH_COL].astype(str).fillna("")
df[NAME_COL] = df[NAME_COL].astype(str).fillna("")
if UNIT_COL not in df.columns: df[UNIT_COL] = "EA"
if "적요" not in df.columns: df["적요"] = ""
df["_uid"] = np.arange(len(df), dtype=int)
in_qty_col = detect_in_qty_col(df)

# (A) 반복 헤더 제거(수불부도 혹시 포함될 수 있어 같이)
df, rep_head_main = drop_repeated_header_rows(df)

# (B) 1년치 블록 중복 제거
df, dup_report = remove_yearly_duplicate_blocks_strict_v2(df, min_block_len=10)

# (C) 본사만 필터 (기존 로직 유지)
df = df[df[WH_COL].astype(str).str.contains(WAREHOUSE_KEYWORD, na=False)].copy()

# (D) 비대상 제거
df, exclude_report = remove_non_product_items(df)

# (E) 중복 제거(일자+품목+거래처+방향) - 기존 플래그 유지
dedup_report = None
if DEDUP_SUFUL_DUPLICATES:
    df, dedup_report = dedup_by_date_item_party_direction(df, in_qty_col)

# (F) 동일일자 재정렬 + 재고 재계산
df = reorder_within_same_day_by_stock_chain(df, in_qty_col)
df = reorder_and_recalc_inventory(df.reset_index(drop=True), in_qty_col)

# ---------- 보세 수불부 전처리 (헤더 반복 제거 중심) ----------
db = df_bonded.copy()
ensure_cols(db, [DATE_COL, ITEM_COL, STOCK_COL, WH_COL, PARTY_COL, NAME_COL, OUT_QTY_COL, OUT_PRICE_COL], ctx="[보세수불부]")
db[DATE_COL] = db[DATE_COL].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
# 먼저 반복헤더 제거
db, rep_head_bonded = drop_repeated_header_rows(db)
db[DATE_COL] = pd.to_datetime(db[DATE_COL], errors="coerce", format="mixed")
db = db.dropna(subset=[DATE_COL]).copy()

db[ITEM_COL] = db[ITEM_COL].astype(str).apply(normalize_item_code)
db[STOCK_COL] = pd.to_numeric(db[STOCK_COL], errors="coerce").fillna(0).astype(float)
db[OUT_QTY_COL] = pd.to_numeric(db[OUT_QTY_COL], errors="coerce").fillna(0).astype(float)
db[OUT_PRICE_COL] = pd.to_numeric(db[OUT_PRICE_COL], errors="coerce").fillna(0).astype(float)
db[PARTY_COL] = db[PARTY_COL].astype(str).fillna("")
db[WH_COL] = db[WH_COL].astype(str).fillna("")
db[NAME_COL] = db[NAME_COL].astype(str).fillna("")
if UNIT_COL not in db.columns: db[UNIT_COL] = "EA"
if "적요" not in db.columns: db["적요"] = ""

out_path = f"{out_dir}/STEP0_전처리_{ts}.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="수불부", index=False)
    df_submit.to_excel(writer, sheet_name="제출본", index=False)
    db.to_excel(writer, sheet_name="보세수불부", index=False)
    pd.DataFrame([{"in_qty_col": in_qty_col}]).to_excel(writer, sheet_name="전처리_in_qty_col", index=False)
    (dup_report if isinstance(dup_report, pd.DataFrame) else pd.DataFrame()).to_excel(writer, sheet_name="전처리_중복1년", index=False)
    (exclude_report if isinstance(exclude_report, pd.DataFrame) else pd.DataFrame()).to_excel(writer, sheet_name="전처리_비대상", index=False)
    (dedup_report if isinstance(dedup_report, pd.DataFrame) else pd.DataFrame([{"dedup":"SKIP"}])).to_excel(writer, sheet_name="전처리_중복(방향)", index=False)
    rep_head_main.to_excel(writer, sheet_name="전처리_헤더반복(수불)", index=False)
    rep_head_bonded.to_excel(writer, sheet_name="전처리_헤더반복(보세)", index=False)
print(f"저장 완료: {out_path}")


 - 수불부 시트: Sheet1
 - 제출본 시트: 본사제출용
 - 보세수불부 시트: 보세수불부_정리


## STEP 1 (기존 STEP0)

In [ ]:

df_suful, df_submit, df_bonded, out_dir = _pick_three_files("STEP 1 (기존 STEP0)")
ts = datetime.now().strftime("%Y%m%d_%H%M%S")

df_submit_template, submit_map, real_map, sys_map, avail_map = init_submit_state(df_submit)
df_submit2 = df_submit_template.copy()
df_submit2[ITEM_COL] = df_submit2[ITEM_COL].astype(str).apply(normalize_item_code)
df_submit2[SUBMIT_USE_COL] = df_submit2[SUBMIT_USE_COL].astype(str).str.strip()
df_submit2[SUBMIT_NAME_COL] = df_submit2[SUBMIT_NAME_COL].astype(str)

df = df_suful.copy()
ensure_cols(df, [DATE_COL, ITEM_COL, STOCK_COL, WH_COL, PARTY_COL, NAME_COL, OUT_QTY_COL, OUT_PRICE_COL], ctx="[수불부]")
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce", format="mixed")
df = df.dropna(subset=[DATE_COL]).copy()
df[ITEM_COL] = df[ITEM_COL].astype(str).apply(normalize_item_code)
df[STOCK_COL] = pd.to_numeric(df[STOCK_COL], errors="coerce").fillna(0).astype(float)
df[OUT_QTY_COL] = pd.to_numeric(df[OUT_QTY_COL], errors="coerce").fillna(0).astype(float)
df[OUT_PRICE_COL] = pd.to_numeric(df[OUT_PRICE_COL], errors="coerce").fillna(0).astype(float)
df[PARTY_COL] = df[PARTY_COL].astype(str).fillna("")
df[WH_COL] = df[WH_COL].astype(str).fillna("")
df[NAME_COL] = df[NAME_COL].astype(str).fillna("")
if UNIT_COL not in df.columns: df[UNIT_COL] = "EA"
if "적요" not in df.columns: df["적요"] = ""
if "_uid" not in df.columns: df["_uid"] = np.arange(len(df), dtype=int)

in_qty_col = detect_in_qty_col(df)
df = reorder_and_recalc_inventory(df.reset_index(drop=True), in_qty_col)
idx0 = FastIndex(df.copy())

year_guess = int(pd.Timestamp(df[DATE_COL].max()).year) if len(df) else datetime.now().year

df1_mod, df_step1_log = apply_step0_adjustments(
    df_pre=df.copy(),
    idx=idx0,
    year_for_jan=year_guess,
    real_map=real_map,
    sys_map=sys_map,
    avail_map=avail_map,
    step0_start_month=1,
    step0_end_month=6,
    verbose=True,
)
df1_mod[STOCK_COL] = idx0.stock
df1_mod = df1_mod.reset_index(drop=True)
df1_mod = reorder_and_recalc_inventory(df1_mod, in_qty_col)

df_submit_step1 = materialize_submit_df(df_submit_template, real_map, sys_map)

out_path = f"{out_dir}/STEP1_기존STEP0_{ts}.xlsx"
_save_step_result(out_path, df1_mod, df_submit_step1, logs={"STEP1_LOG(기존STEP0)": df_step1_log})


## STEP 2 (기존 STEP0-1)

In [ ]:

# =========================================================
# STEP 2 (기존 STEP0-1): 큰 출고 분할/미루기(대여)
# - 입력: STEP1 결과의 수불부/제출본/보세수불부
# - 출력: STEP2_결과.xlsx
# =========================================================
df_suful, df_submit, df_bonded, out_dir = _pick_three_files("STEP 2 (기존 STEP0-1)")
ts = datetime.now().strftime("%Y%m%d_%H%M%S")

# 제출본 상태
df_submit_template, submit_map, real_map, sys_map, avail_map = init_submit_state(df_submit)

df = df_suful.copy()
ensure_cols(df, [DATE_COL, ITEM_COL, STOCK_COL, WH_COL, PARTY_COL, NAME_COL, OUT_QTY_COL, OUT_PRICE_COL], ctx="[수불부]")
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce", format="mixed")
df = df.dropna(subset=[DATE_COL]).copy()
df[ITEM_COL] = df[ITEM_COL].astype(str).apply(normalize_item_code)
df[STOCK_COL] = pd.to_numeric(df[STOCK_COL], errors="coerce").fillna(0).astype(float)
df[OUT_QTY_COL] = pd.to_numeric(df[OUT_QTY_COL], errors="coerce").fillna(0).astype(float)
df[OUT_PRICE_COL] = pd.to_numeric(df[OUT_PRICE_COL], errors="coerce").fillna(0).astype(float)
df[PARTY_COL] = df[PARTY_COL].astype(str).fillna("")
df[WH_COL] = df[WH_COL].astype(str).fillna("")
df[NAME_COL] = df[NAME_COL].astype(str).fillna("")
if UNIT_COL not in df.columns: df[UNIT_COL]="EA"
if "적요" not in df.columns: df["적요"]=""

# in_qty 추정 + 재계산 기반 정렬
in_qty_col = detect_in_qty_col(df)
df = reorder_and_recalc_inventory(df.reset_index(drop=True), in_qty_col)
idx = FastIndex(df)

problem_items = get_problem_items(df, idx)

df2_mod, df_step2_log = apply_step0_1_split_outbound(
    df=df,
    problem_items=problem_items,
    worst_threshold=STEP01_WORST_THRESHOLD,
    max_future_months=STEP01_MAX_FUTURE_MONTHS,
    min_big_sale_ratio=STEP01_MIN_BIG_SALE_RATIO,
    exclude_party_keywords=STEP01_EXCLUDE_PARTY_KEYWORDS,
    verbose=True,
)
df2_mod = df2_mod.reset_index(drop=True)
df2_mod = reorder_and_recalc_inventory(df2_mod, in_qty_col)

df_submit_step2 = materialize_submit_df(df_submit_template, real_map, sys_map)

out_path = f"{out_dir}/STEP2_기존STEP0-1_{ts}.xlsx"
_save_step_result(
    out_path,
    df2_mod,
    df_submit_step2,
    logs={"STEP2_LOG(기존STEP0-1)": df_step2_log},
)


## STEP 3 (기존 STEP00)

In [ ]:

# =========================================================
# STEP 3 (기존 STEP00): 입고 당김 + 보세 룰
# - 입력: STEP2 결과의 수불부/제출본/보세수불부
# - 출력: STEP3_결과.xlsx (수불부/제출본/보세수불부 수정본 포함)
# =========================================================
df_suful, df_submit, df_bonded, out_dir = _pick_three_files("STEP 3 (기존 STEP00)")
ts = datetime.now().strftime("%Y%m%d_%H%M%S")

# 제출본 상태
df_submit_template, submit_map, real_map, sys_map, avail_map = init_submit_state(df_submit)

df_main = df_suful.copy()
ensure_cols(df_main, [DATE_COL, ITEM_COL, STOCK_COL, WH_COL, PARTY_COL, NAME_COL, OUT_QTY_COL, OUT_PRICE_COL], ctx="[수불부]")
df_main[DATE_COL] = pd.to_datetime(df_main[DATE_COL], errors="coerce", format="mixed")
df_main = df_main.dropna(subset=[DATE_COL]).copy()
df_main[ITEM_COL] = df_main[ITEM_COL].astype(str).apply(normalize_item_code)
df_main[STOCK_COL] = pd.to_numeric(df_main[STOCK_COL], errors="coerce").fillna(0).astype(float)
df_main[OUT_QTY_COL] = pd.to_numeric(df_main[OUT_QTY_COL], errors="coerce").fillna(0).astype(float)
df_main[OUT_PRICE_COL] = pd.to_numeric(df_main[OUT_PRICE_COL], errors="coerce").fillna(0).astype(float)
df_main[PARTY_COL] = df_main[PARTY_COL].astype(str).fillna("")
df_main[WH_COL] = df_main[WH_COL].astype(str).fillna("")
df_main[NAME_COL] = df_main[NAME_COL].astype(str).fillna("")
if UNIT_COL not in df_main.columns: df_main[UNIT_COL]="EA"
if "적요" not in df_main.columns: df_main["적요"]=""

df_b = df_bonded.copy()
ensure_cols(df_b, [DATE_COL, ITEM_COL, STOCK_COL, WH_COL, PARTY_COL, NAME_COL, OUT_QTY_COL, OUT_PRICE_COL], ctx="[보세수불부]")
df_b[DATE_COL] = pd.to_datetime(df_b[DATE_COL], errors="coerce", format="mixed")
df_b = df_b.dropna(subset=[DATE_COL]).copy()
df_b[ITEM_COL] = df_b[ITEM_COL].astype(str).apply(normalize_item_code)
df_b[STOCK_COL] = pd.to_numeric(df_b[STOCK_COL], errors="coerce").fillna(0).astype(float)
df_b[OUT_QTY_COL] = pd.to_numeric(df_b[OUT_QTY_COL], errors="coerce").fillna(0).astype(float)
df_b[OUT_PRICE_COL] = pd.to_numeric(df_b[OUT_PRICE_COL], errors="coerce").fillna(0).astype(float)
df_b[PARTY_COL] = df_b[PARTY_COL].astype(str).fillna("")
df_b[WH_COL] = df_b[WH_COL].astype(str).fillna("")
df_b[NAME_COL] = df_b[NAME_COL].astype(str).fillna("")
if UNIT_COL not in df_b.columns: df_b[UNIT_COL]="EA"
if "적요" not in df_b.columns: df_b["적요"]=""

in_qty_col_main = detect_in_qty_col(df_main)
in_qty_col_bonded = detect_in_qty_col(df_b)

df_main = reorder_and_recalc_inventory(df_main.reset_index(drop=True), in_qty_col_main)
idx = FastIndex(df_main)
problem_items = get_problem_items(df_main, idx)

df3_mod, df_b_mod, df_step3_main_log, df_step3_bonded_log = apply_step00_pull_inbound_with_bonded(
    df_main=df_main,
    df_bonded=df_b,
    idx_main=idx,
    problem_items=problem_items,
    in_qty_col_main=in_qty_col_main,
    in_qty_col_bonded=in_qty_col_bonded,
    worst_threshold=STEP00_WORST_THRESHOLD,
    allow_cross_month=STEP00_ALLOW_CROSS_MONTH,
    max_pull_days=STEP00_MAX_PULL_DAYS,
    max_pull_months=STEP00_MAX_PULL_MONTHS,
    verbose=True,
)

df3_mod = df3_mod.reset_index(drop=True)
df3_mod = reorder_and_recalc_inventory(df3_mod, in_qty_col_main)

df_submit_step3 = materialize_submit_df(df_submit_template, real_map, sys_map)

out_path = f"{out_dir}/STEP3_기존STEP00_{ts}.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    df3_mod.to_excel(writer, sheet_name="수불부", index=False)
    df_submit_step3.to_excel(writer, sheet_name="제출본", index=False)
    df_b_mod.to_excel(writer, sheet_name="보세수불부", index=False)
    df_step3_main_log.to_excel(writer, sheet_name="STEP3_LOG_MAIN", index=False)
    df_step3_bonded_log.to_excel(writer, sheet_name="STEP3_LOG_BONDED", index=False)
print(f"저장 완료: {out_path}")


## STEP 4 (기존 STEP1~3 통합)

In [ ]:

df_suful, df_submit, df_bonded, out_dir = _pick_three_files("STEP 4 (기존 STEP1~3 통합)")
ts = datetime.now().strftime("%Y%m%d_%H%M%S")

df_submit_template, submit_map, real_map, sys_map, avail_map = init_submit_state(df_submit)

df_submit2 = df_submit_template.copy()
df_submit2[ITEM_COL] = df_submit2[ITEM_COL].astype(str).apply(normalize_item_code)
df_submit2[SUBMIT_USE_COL] = df_submit2[SUBMIT_USE_COL].astype(str).str.strip()
df_submit2[SUBMIT_NAME_COL] = df_submit2[SUBMIT_NAME_COL].astype(str)

df = df_suful.copy()
ensure_cols(df, [DATE_COL, ITEM_COL, STOCK_COL, WH_COL, PARTY_COL, NAME_COL, OUT_QTY_COL, OUT_PRICE_COL], ctx="[수불부]")
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce", format="mixed")
df = df.dropna(subset=[DATE_COL]).copy()
df[ITEM_COL] = df[ITEM_COL].astype(str).apply(normalize_item_code)
df[STOCK_COL] = pd.to_numeric(df[STOCK_COL], errors="coerce").fillna(0).astype(float)
df[OUT_QTY_COL] = pd.to_numeric(df[OUT_QTY_COL], errors="coerce").fillna(0).astype(float)
df[OUT_PRICE_COL] = pd.to_numeric(df[OUT_PRICE_COL], errors="coerce").fillna(0).astype(float)
df[PARTY_COL] = df[PARTY_COL].astype(str).fillna("")
df[WH_COL] = df[WH_COL].astype(str).fillna("")
df[NAME_COL] = df[NAME_COL].astype(str).fillna("")
if UNIT_COL not in df.columns: df[UNIT_COL] = "EA"
if "적요" not in df.columns: df["적요"] = ""
if "_uid" not in df.columns: df["_uid"] = np.arange(len(df), dtype=int)

in_qty_col = detect_in_qty_col(df)
df = reorder_and_recalc_inventory(df.reset_index(drop=True), in_qty_col)

yearly_min_stock_map, yearly_min_stock_df = build_yearly_min_stock_table(df)

df4_mod, df_move_log, problem_after3 = apply_step1_to3_combined(
    df_in=df,
    submit_map=submit_map,
    df_submit2=df_submit2,
    real_map=real_map,
    sys_map=sys_map,
    avail_map=avail_map,
    yearly_min_stock_map=yearly_min_stock_map,
)

df_submit_step4 = materialize_submit_df(df_submit_template, real_map, sys_map)

out_path = f"{out_dir}/STEP4_기존STEP1~3_{ts}.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    df4_mod.to_excel(writer, sheet_name="수불부", index=False)
    df_submit_step4.to_excel(writer, sheet_name="제출본", index=False)
    yearly_min_stock_df.to_excel(writer, sheet_name="연중최소재고", index=False)
    df_move_log.to_excel(writer, sheet_name="이동로그", index=False)
    pd.DataFrame([{"남은문제품목수": len(problem_after3)}]).to_excel(writer, sheet_name="요약", index=False)
print(f"저장 완료: {out_path}")


## STEP 5 (기존 STEP4)

In [ ]:

# =========================================================
# STEP 5 (기존 STEP4): 동일 품목 내 프로모(10+1) 보정
# - 입력: STEP4 결과의 수불부/제출본/보세수불부
# - 출력: STEP5_결과.xlsx
# =========================================================
df_suful, df_submit, df_bonded, out_dir = _pick_three_files("STEP 5 (기존 STEP4)")
ts = datetime.now().strftime("%Y%m%d_%H%M%S")

df_submit_template, submit_map, real_map, sys_map, avail_map = init_submit_state(df_submit)

df = df_suful.copy()
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce", format="mixed")
df = df.dropna(subset=[DATE_COL]).copy()
df[ITEM_COL] = df[ITEM_COL].astype(str).apply(normalize_item_code)
df[STOCK_COL] = pd.to_numeric(df[STOCK_COL], errors="coerce").fillna(0).astype(float)
df[OUT_QTY_COL] = pd.to_numeric(df[OUT_QTY_COL], errors="coerce").fillna(0).astype(float)
df[OUT_PRICE_COL] = pd.to_numeric(df[OUT_PRICE_COL], errors="coerce").fillna(0).astype(float)
df[PARTY_COL] = df[PARTY_COL].astype(str).fillna("")
df[WH_COL] = df[WH_COL].astype(str).fillna("")
df[NAME_COL] = df[NAME_COL].astype(str).fillna("")
if UNIT_COL not in df.columns: df[UNIT_COL]="EA"
if "적요" not in df.columns: df["적요"]=""

in_qty_col = detect_in_qty_col(df)
df = reorder_and_recalc_inventory(df.reset_index(drop=True), in_qty_col)
idx = FastIndex(df)
problem_items = get_problem_items(df, idx)

df5_mod, df_step5_log = apply_step4_same_item_promo(
    df=df,
    idx=idx,
    problem_items=problem_items,
    in_qty_col=in_qty_col,
    verbose=True,
)
df5_mod[STOCK_COL] = idx.stock
df5_mod = df5_mod.reset_index(drop=True)
df5_mod = reorder_and_recalc_inventory(df5_mod, in_qty_col)

df_submit_step5 = materialize_submit_df(df_submit_template, real_map, sys_map)

out_path = f"{out_dir}/STEP5_기존STEP4_{ts}.xlsx"
_save_step_result(out_path, df5_mod, df_submit_step5, logs={"STEP5_LOG(기존STEP4)": df_step5_log})


## STEP 6 (기존 STEP5)

In [ ]:

# =========================================================
# STEP 6 (기존 STEP5): 단가 보정 (출고단가 없는 10+1 등)
# - 입력: STEP5 결과의 수불부/제출본/보세수불부
# - 출력: STEP6_결과.xlsx
# =========================================================
df_suful, df_submit, df_bonded, out_dir = _pick_three_files("STEP 6 (기존 STEP5)")
ts = datetime.now().strftime("%Y%m%d_%H%M%S")

df_submit_template, submit_map, real_map, sys_map, avail_map = init_submit_state(df_submit)

df = df_suful.copy()
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce", format="mixed")
df = df.dropna(subset=[DATE_COL]).copy()
df[ITEM_COL] = df[ITEM_COL].astype(str).apply(normalize_item_code)
df[STOCK_COL] = pd.to_numeric(df[STOCK_COL], errors="coerce").fillna(0).astype(float)
df[OUT_QTY_COL] = pd.to_numeric(df[OUT_QTY_COL], errors="coerce").fillna(0).astype(float)
df[OUT_PRICE_COL] = pd.to_numeric(df[OUT_PRICE_COL], errors="coerce").fillna(0).astype(float)
df[PARTY_COL] = df[PARTY_COL].astype(str).fillna("")
df[WH_COL] = df[WH_COL].astype(str).fillna("")
df[NAME_COL] = df[NAME_COL].astype(str).fillna("")
if UNIT_COL not in df.columns: df[UNIT_COL]="EA"
if "적요" not in df.columns: df["적요"]=""

in_qty_col = detect_in_qty_col(df)
df = reorder_and_recalc_inventory(df.reset_index(drop=True), in_qty_col)
idx = FastIndex(df)
pricebook = PriceBook(df)

df6_mod, df_step6_log = apply_step5_reprice_sales(
    df=df,
    idx=idx,
    pricebook=pricebook,
    in_qty_col=in_qty_col,
    verbose=True,
)
df6_mod[STOCK_COL] = idx.stock
df6_mod = df6_mod.reset_index(drop=True)
df6_mod = reorder_and_recalc_inventory(df6_mod, in_qty_col)

df_submit_step6 = materialize_submit_df(df_submit_template, real_map, sys_map)

out_path = f"{out_dir}/STEP6_기존STEP5_{ts}.xlsx"
_save_step_result(out_path, df6_mod, df_submit_step6, logs={"STEP6_LOG(기존STEP5)": df_step6_log})
